In [1]:
# ============================================================
# CELL 1 — LOCKED EXPERIMENT CONTRACT
# External SFUDA Baseline Comparison
#
# IMPORTANT:
# This cell defines the shared experimental setup.
# Do not change these values after baseline experiments begin.
# ============================================================

import os
import random
import json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

# ------------------------------------------------------------
# Dataset paths — LOCKED
# ------------------------------------------------------------
IRISH_ROOT = Path("/kaggle/input/datasets/manhhoangvan/irishpotato")
SOURCE_ROOT = Path("/kaggle/input/datasets/faysalmiah1721758/potato-dataset")
PLD_ROOT = Path("/kaggle/input/datasets/shahadhossin567r7455/potato-leaf-disease-dataset")

INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")

# ------------------------------------------------------------
# Experimental setting — LOCKED
# ------------------------------------------------------------
CLASS_NAMES = [
    "Early Blight",
    "Late Blight",
    "Healthy"
]

CLASS_TO_ID = {
    "Early Blight": 0,
    "Late Blight": 1,
    "Healthy": 2
}

ID_TO_CLASS = {
    0: "Early Blight",
    1: "Late Blight",
    2: "Healthy"
}

NUM_CLASSES = 3

SEEDS = [42, 123, 2024]

IMG_SIZE = 224
BATCH_SIZE = 32

BACKBONE = "ResNet50"
PRETRAINED_WEIGHTS = "imagenet"

# ------------------------------------------------------------
# Source split — LOCKED
# ------------------------------------------------------------
SOURCE_TRAIN_RATIO = 0.70
SOURCE_VAL_RATIO = 0.15
SOURCE_TEST_RATIO = 0.15

# ------------------------------------------------------------
# Hard-target construction — LOCKED
# ------------------------------------------------------------
HARD_TARGET_PER_CLASS = 3000

# 80/20 after balanced sampling
HARD_TARGET_TRAIN_RATIO = 0.80
HARD_TARGET_TEST_RATIO = 0.20

EXPECTED_HARD_TARGET_TOTAL = 9000
EXPECTED_HARD_TARGET_TRAIN = 7200
EXPECTED_HARD_TARGET_TEST = 1800

# ------------------------------------------------------------
# Mild-target construction — LOCKED
# ------------------------------------------------------------
MILD_TARGET_TEST_RATIO = 0.30

EXPECTED_MILD_TOTAL = 1500
EXPECTED_MILD_TRAIN = 1050
EXPECTED_MILD_TEST = 450

# ------------------------------------------------------------
# OOD evaluation — LOCKED
# ------------------------------------------------------------
EXPECTED_OOD_TOTAL = 2000

# ------------------------------------------------------------
# Evaluation metrics — LOCKED
# ------------------------------------------------------------
METRICS = [
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "ece",
    "brier",
    "nll"
]

ECE_BINS = 15

# ------------------------------------------------------------
# Important experimental constraints
# ------------------------------------------------------------
TARGET_TEST_LABELS_USED_FOR_TRAINING = False
OOD_LABELS_USED_FOR_TRAINING = False

# The target training set is treated as unlabeled by adaptation methods.
TARGET_TRAINING_IS_UNLABELED = True

# ------------------------------------------------------------
# Reference results from completed experiment
# Used only as reproduction checks.
# ------------------------------------------------------------
REFERENCE_SOURCE_ONLY_HARD_ACC = {
    42: 0.813333,
    123: 0.843333,
    2024: 0.833333
}

REFERENCE_PROPOSED_HARD_ACC = {
    42: 0.832222,
    123: 0.845000,
    2024: 0.858333
}

# ------------------------------------------------------------
# Proposed-method settings
# Stored for documentation only.
# External baselines must NOT use these unless their method requires them.
# ------------------------------------------------------------
PROPOSED_CONF_ACCEPT = 0.9999
PROPOSED_CONF_REJECT = 0.6667
PROPOSED_DEFER_TEMPERATURE = 2.0
PROPOSED_ACCEPT_WEIGHT = 1.0
PROPOSED_DEFER_WEIGHT = 0.3
PROPOSED_CLASS_WEIGHT_CAP = 3.0

# ------------------------------------------------------------
# Reproducibility helper
# ------------------------------------------------------------
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

print("=" * 72)
print("LOCKED EXTERNAL-BASELINE EXPERIMENT CONTRACT")
print("=" * 72)

print("Datasets:")
print(" Irish Potato :", IRISH_ROOT)
print(" Source       :", SOURCE_ROOT)
print(" PLD          :", PLD_ROOT)

print("\nBackbone:", BACKBONE)
print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Seeds:", SEEDS)

print("\nClass order:")
for k, v in CLASS_TO_ID.items():
    print(f"  {v}: {k}")

print("\nExpected hard-target split:")
print(" Total :", EXPECTED_HARD_TARGET_TOTAL)
print(" Train :", EXPECTED_HARD_TARGET_TRAIN)
print(" Test  :", EXPECTED_HARD_TARGET_TEST)

print("\nContract loaded successfully.")

LOCKED EXTERNAL-BASELINE EXPERIMENT CONTRACT
Datasets:
 Irish Potato : /kaggle/input/datasets/manhhoangvan/irishpotato
 Source       : /kaggle/input/datasets/faysalmiah1721758/potato-dataset
 PLD          : /kaggle/input/datasets/shahadhossin567r7455/potato-leaf-disease-dataset

Backbone: ResNet50
Image size: 224
Batch size: 32
Seeds: [42, 123, 2024]

Class order:
  0: Early Blight
  1: Late Blight
  2: Healthy

Expected hard-target split:
 Total : 9000
 Train : 7200
 Test  : 1800

Contract loaded successfully.


In [2]:
# ============================================================
# CELL 2 — DATASET SCAN + MANIFEST VERIFICATION
# No training. No splitting. No adaptation.
# ============================================================

from collections import Counter

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# ------------------------------------------------------------
# Helper: collect image files recursively
# ------------------------------------------------------------
def collect_images(root):
    files = []
    for p in Path(root).rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            files.append(p)
    return sorted(files)

# ------------------------------------------------------------
# Explicit class-name normalization
# ------------------------------------------------------------
def normalize_name(name):
    return (
        name.strip()
        .lower()
        .replace("_", " ")
        .replace("-", " ")
        .replace(".", " ")
    )

def canonical_closed_set_class(name):
    n = normalize_name(name)

    # Irish Potato dataset folder names
    if "earlyblt" in n:
        return "Early Blight"

    if "lateblt" in n:
        return "Late Blight"

    # Other datasets
    if "early" in n and "blight" in n:
        return "Early Blight"

    if "late" in n and "blight" in n:
        return "Late Blight"

    if "healthy" in n:
        return "Healthy"

    return None

# ------------------------------------------------------------
# Dataset-specific manifest builder
# ------------------------------------------------------------
def build_manifest(root, dataset_name):
    image_files = collect_images(root)

    rows = []

    for fp in image_files:
        # inspect all parent folders, nearest first
        parents = [p.name for p in fp.parents if p != Path(root).parent]

        canonical = None
        raw_class = None

        for parent_name in parents:
            c = canonical_closed_set_class(parent_name)
            if c is not None:
                canonical = c
                raw_class = parent_name
                break

        rows.append({
            "dataset": dataset_name,
            "file_path": str(fp),
            "raw_class": raw_class,
            "canonical_class": canonical
        })

    return pd.DataFrame(rows)

# ------------------------------------------------------------
# Build manifests
# ------------------------------------------------------------
df_source_all = build_manifest(
    SOURCE_ROOT,
    "source_clean"
)

df_pld_all = build_manifest(
    PLD_ROOT,
    "pld"
)

df_irish_all = build_manifest(
    IRISH_ROOT,
    "irish_potato"
)

print("=" * 72)
print("RAW IMAGE COUNTS")
print("=" * 72)

print("Source images :", len(df_source_all))
print("PLD images    :", len(df_pld_all))
print("Irish images  :", len(df_irish_all))

# ------------------------------------------------------------
# Closed-set manifests
# ------------------------------------------------------------
df_source_closed = df_source_all[
    df_source_all["canonical_class"].notna()
].copy()

df_pld_closed = df_pld_all[
    df_pld_all["canonical_class"].notna()
].copy()

df_irish_closed = df_irish_all[
    df_irish_all["canonical_class"].notna()
].copy()

# ------------------------------------------------------------
# Identify PLD OOD images
# Any image that is NOT one of the three known classes
# ------------------------------------------------------------
df_pld_ood = df_pld_all[
    df_pld_all["canonical_class"].isna()
].copy()

# use nearest folder containing useful class identity
def infer_nearest_class_folder(fp, dataset_root):
    fp = Path(fp)
    root = Path(dataset_root)

    relative_parts = fp.relative_to(root).parts

    # remove filename
    folders = relative_parts[:-1]

    if not folders:
        return "UNKNOWN"

    return folders[-1]

df_pld_ood["raw_ood_class"] = df_pld_ood["file_path"].apply(
    lambda x: infer_nearest_class_folder(x, PLD_ROOT)
)

# ------------------------------------------------------------
# Reporting helper
# ------------------------------------------------------------
def print_class_counts(df, title):
    print("\n" + title)
    print("-" * len(title))

    counts = (
        df["canonical_class"]
        .value_counts()
        .reindex(CLASS_NAMES)
        .fillna(0)
        .astype(int)
    )

    for cls, count in counts.items():
        print(f"{cls:15s}: {count}")

    print(f"{'Total':15s}: {counts.sum()}")

    return counts

source_counts = print_class_counts(
    df_source_closed,
    "SOURCE CLOSED-SET COUNTS"
)

pld_counts = print_class_counts(
    df_pld_closed,
    "PLD CLOSED-SET COUNTS"
)

irish_counts = print_class_counts(
    df_irish_closed,
    "IRISH POTATO CLOSED-SET COUNTS"
)

# ------------------------------------------------------------
# OOD reporting
# ------------------------------------------------------------
print("\nPLD OOD COUNTS")
print("--------------")

ood_counts = (
    df_pld_ood["raw_ood_class"]
    .value_counts()
    .sort_index()
)

print(ood_counts)
print("Total OOD:", len(df_pld_ood))

# ------------------------------------------------------------
# Expected-count assertions
# ------------------------------------------------------------

EXPECTED_SOURCE_COUNTS = {
    "Early Blight": 1000,
    "Late Blight": 1000,
    "Healthy": 152
}

EXPECTED_PLD_COUNTS = {
    "Early Blight": 500,
    "Late Blight": 500,
    "Healthy": 500
}

EXPECTED_IRISH_COUNTS = {
    "Early Blight": 17772,
    "Late Blight": 20499,
    "Healthy": 20438
}

for cls, expected in EXPECTED_SOURCE_COUNTS.items():
    actual = int(source_counts.loc[cls])
    assert actual == expected, (
        f"Source count mismatch for {cls}: "
        f"expected {expected}, got {actual}"
    )

for cls, expected in EXPECTED_PLD_COUNTS.items():
    actual = int(pld_counts.loc[cls])
    assert actual == expected, (
        f"PLD count mismatch for {cls}: "
        f"expected {expected}, got {actual}"
    )

for cls, expected in EXPECTED_IRISH_COUNTS.items():
    actual = int(irish_counts.loc[cls])
    assert actual == expected, (
        f"Irish count mismatch for {cls}: "
        f"expected {expected}, got {actual}"
    )

assert len(df_source_closed) == 2152, (
    f"Expected 2152 source images, got {len(df_source_closed)}"
)

assert len(df_pld_closed) == EXPECTED_MILD_TOTAL, (
    f"Expected {EXPECTED_MILD_TOTAL} PLD closed-set images, "
    f"got {len(df_pld_closed)}"
)

assert len(df_irish_closed) == 58709, (
    f"Expected 58709 Irish images, got {len(df_irish_closed)}"
)

assert len(df_pld_ood) == EXPECTED_OOD_TOTAL, (
    f"Expected {EXPECTED_OOD_TOTAL} OOD images, "
    f"got {len(df_pld_ood)}"
)

print("\n" + "=" * 72)
print("DATASET MANIFEST CHECK PASSED")
print("=" * 72)

print("Source closed-set :", len(df_source_closed))
print("PLD closed-set    :", len(df_pld_closed))
print("PLD OOD           :", len(df_pld_ood))
print("Irish closed-set  :", len(df_irish_closed))

RAW IMAGE COUNTS
Source images : 2152
PLD images    : 3500
Irish images  : 58709

SOURCE CLOSED-SET COUNTS
------------------------
Early Blight   : 1000
Late Blight    : 1000
Healthy        : 152
Total          : 2152

PLD CLOSED-SET COUNTS
---------------------
Early Blight   : 500
Late Blight    : 500
Healthy        : 500
Total          : 1500

IRISH POTATO CLOSED-SET COUNTS
------------------------------
Early Blight   : 17772
Late Blight    : 20499
Healthy        : 20438
Total          : 58709

PLD OOD COUNTS
--------------
raw_ood_class
Fungal Diseases         500
Plant Pests             500
Potato Cyst Nematode    500
Potato Virus            500
Name: count, dtype: int64
Total OOD: 2000

DATASET MANIFEST CHECK PASSED
Source closed-set : 2152
PLD closed-set    : 1500
PLD OOD           : 2000
Irish closed-set  : 58709


In [3]:
# ============================================================
# CELL 3 — LOCKED SEED-SPECIFIC SPLITS
# Reproduces the final multi-seed split logic from the old notebook.
# No model training yet.
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# ------------------------------------------------------------
# Add integer labels
# ------------------------------------------------------------
def add_label_id(df):
    df = df.copy()
    df["label"] = df["canonical_class"].map(CLASS_TO_ID)

    assert df["label"].notna().all(), "Found unmapped closed-set class."
    df["label"] = df["label"].astype(int)

    return df

source_base_df = add_label_id(df_source_closed)
target_mild_base_df = add_label_id(df_pld_closed)
target_hard_base_df = add_label_id(df_irish_closed)

# OOD remains outside the 3-class label space
ood_test_df = df_pld_ood.copy()

# ------------------------------------------------------------
# Exact split helpers from the completed experiment
# ------------------------------------------------------------
def stratified_split(df, test_size, seed):
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df["label"],
        random_state=seed
    )

    return (
        train_df.reset_index(drop=True),
        test_df.reset_index(drop=True)
    )

def balanced_sample_no_warning(df, per_class, seed):
    sampled_parts = []

    for label_id, sub in df.groupby("label"):
        n = min(per_class, len(sub))

        sampled_parts.append(
            sub.sample(
                n=n,
                random_state=seed
            )
        )

    return pd.concat(
        sampled_parts,
        ignore_index=True
    )

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------
SPLIT_ROOT = WORK_DIR / "external_baseline_locked_splits"
SPLIT_ROOT.mkdir(parents=True, exist_ok=True)

all_seed_splits = {}

# ------------------------------------------------------------
# Build exactly the same split structure for each seed
# ------------------------------------------------------------
for seed in SEEDS:

    seed_dir = SPLIT_ROOT / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)

    # ========================================================
    # SOURCE: 70 / 15 / 15
    # ========================================================

    source_train, source_temp = stratified_split(
        source_base_df,
        test_size=0.30,
        seed=seed
    )

    source_val, source_test = stratified_split(
        source_temp,
        test_size=0.50,
        seed=seed
    )

    # ========================================================
    # MILD TARGET: 70 / 30
    # ========================================================

    target_mild_train, target_mild_test = stratified_split(
        target_mild_base_df,
        test_size=0.30,
        seed=seed
    )

    # ========================================================
    # HARD TARGET:
    # 1. sample exactly 3000 per class
    # 2. split 80 / 20
    # ========================================================

    target_hard_balanced = balanced_sample_no_warning(
        target_hard_base_df,
        per_class=HARD_TARGET_PER_CLASS,
        seed=seed
    )

    target_hard_train, target_hard_test = stratified_split(
        target_hard_balanced,
        test_size=0.20,
        seed=seed
    )

    # ========================================================
    # Source class weights
    # Same computation as completed experiment
    # ========================================================

    classes = np.array([0, 1, 2])

    class_weights_array = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=source_train["label"].values
    )

    class_weights = {
        int(cls): float(weight)
        for cls, weight in zip(classes, class_weights_array)
    }

    # ========================================================
    # Save split files
    # ========================================================

    source_train.to_csv(
        seed_dir / "source_train.csv",
        index=False
    )

    source_val.to_csv(
        seed_dir / "source_val.csv",
        index=False
    )

    source_test.to_csv(
        seed_dir / "source_test.csv",
        index=False
    )

    target_mild_train.to_csv(
        seed_dir / "target_mild_train_unlabeled.csv",
        index=False
    )

    target_mild_test.to_csv(
        seed_dir / "target_mild_test_labeled.csv",
        index=False
    )

    target_hard_train.to_csv(
        seed_dir / "target_hard_train_unlabeled.csv",
        index=False
    )

    target_hard_test.to_csv(
        seed_dir / "target_hard_test_labeled.csv",
        index=False
    )

    # Same OOD set for every seed
    ood_test_df.to_csv(
        seed_dir / "ood_extra_test.csv",
        index=False
    )

    # --------------------------------------------------------
    # Store in memory
    # --------------------------------------------------------

    all_seed_splits[seed] = {
        "source_train": source_train,
        "source_val": source_val,
        "source_test": source_test,
        "target_mild_train": target_mild_train,
        "target_mild_test": target_mild_test,
        "target_hard_balanced": target_hard_balanced,
        "target_hard_train": target_hard_train,
        "target_hard_test": target_hard_test,
        "ood_test": ood_test_df.copy(),
        "class_weights": class_weights
    }

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

def count_by_class(df):
    return (
        df["canonical_class"]
        .value_counts()
        .reindex(CLASS_NAMES)
        .fillna(0)
        .astype(int)
    )

print("=" * 76)
print("LOCKED MULTI-SEED SPLIT VERIFICATION")
print("=" * 76)

for seed in SEEDS:

    s = all_seed_splits[seed]

    print(f"\n{'='*30} SEED {seed} {'='*30}")

    print("\nSource train:")
    print(count_by_class(s["source_train"]).to_string())

    print("\nSource val:")
    print(count_by_class(s["source_val"]).to_string())

    print("\nSource test:")
    print(count_by_class(s["source_test"]).to_string())

    print("\nMild target train:")
    print(count_by_class(s["target_mild_train"]).to_string())

    print("\nMild target test:")
    print(count_by_class(s["target_mild_test"]).to_string())

    print("\nHard target train:")
    print(count_by_class(s["target_hard_train"]).to_string())

    print("\nHard target test:")
    print(count_by_class(s["target_hard_test"]).to_string())

    print("\nSource class weights:")
    print(s["class_weights"])

    # --------------------------------------------------------
    # Locked assertions
    # --------------------------------------------------------

    assert len(s["source_train"]) == 1506
    assert len(s["source_val"]) == 323
    assert len(s["source_test"]) == 323

    assert len(s["target_mild_train"]) == 1050
    assert len(s["target_mild_test"]) == 450

    assert len(s["target_hard_balanced"]) == 9000
    assert len(s["target_hard_train"]) == 7200
    assert len(s["target_hard_test"]) == 1800

    assert len(s["ood_test"]) == 2000

    # Hard target must be perfectly balanced
    hard_train_counts = count_by_class(
        s["target_hard_train"]
    )

    hard_test_counts = count_by_class(
        s["target_hard_test"]
    )

    assert (hard_train_counts.values == 2400).all()
    assert (hard_test_counts.values == 600).all()

print("\n" + "=" * 76)
print("ALL SEED-SPECIFIC SPLITS PASSED")
print("=" * 76)

print("Saved to:")
print(SPLIT_ROOT)

LOCKED MULTI-SEED SPLIT VERIFICATION

============================== SEED 42 ==============================

Source train:
canonical_class
Early Blight    700
Late Blight     700
Healthy         106

Source val:
canonical_class
Early Blight    150
Late Blight     150
Healthy          23

Source test:
canonical_class
Early Blight    150
Late Blight     150
Healthy          23

Mild target train:
canonical_class
Early Blight    350
Late Blight     350
Healthy         350

Mild target test:
canonical_class
Early Blight    150
Late Blight     150
Healthy         150

Hard target train:
canonical_class
Early Blight    2400
Late Blight     2400
Healthy         2400

Hard target test:
canonical_class
Early Blight    600
Late Blight     600
Healthy         600

Source class weights:
{0: 0.7171428571428572, 1: 0.7171428571428572, 2: 4.735849056603773}

============================== SEED 123 ==============================

Source train:
canonical_class
Early Blight    700
Late Blight     700
He

In [4]:
# ============================================================
# CELL 4 — SPLIT INTEGRITY + IMAGE VALIDITY CHECKS
#
# Goals:
# 1. Verify no train/test overlap within each seed.
# 2. Verify the source/target/OOD sets remain disjoint by path.
# 3. Scan hard-target training/test images for decoding failures.
# 4. Reproduce the known Seed 2024 bad-image handling.
#
# NO MODEL TRAINING HERE.
# ============================================================

import hashlib
from IPython.display import display

# ------------------------------------------------------------
# Helper: path-set overlap
# ------------------------------------------------------------
def path_set(df):
    return set(df["file_path"].astype(str).tolist())

def overlap_count(df_a, df_b):
    return len(path_set(df_a) & path_set(df_b))

def assert_no_overlap(df_a, df_b, name_a, name_b):
    overlap = path_set(df_a) & path_set(df_b)

    assert len(overlap) == 0, (
        f"Leakage detected between {name_a} and {name_b}: "
        f"{len(overlap)} overlapping files."
    )

# ------------------------------------------------------------
# 1. Split integrity checks
# ------------------------------------------------------------
print("=" * 76)
print("SPLIT INTEGRITY CHECKS")
print("=" * 76)

for seed in SEEDS:
    s = all_seed_splits[seed]

    print(f"\nSeed {seed}")

    # Source
    assert_no_overlap(
        s["source_train"],
        s["source_val"],
        "source_train",
        "source_val"
    )

    assert_no_overlap(
        s["source_train"],
        s["source_test"],
        "source_train",
        "source_test"
    )

    assert_no_overlap(
        s["source_val"],
        s["source_test"],
        "source_val",
        "source_test"
    )

    # Mild target
    assert_no_overlap(
        s["target_mild_train"],
        s["target_mild_test"],
        "target_mild_train",
        "target_mild_test"
    )

    # Hard target
    assert_no_overlap(
        s["target_hard_train"],
        s["target_hard_test"],
        "target_hard_train",
        "target_hard_test"
    )

    print("  Source train/val/test overlap : 0")
    print("  Mild target train/test overlap: 0")
    print("  Hard target train/test overlap: 0")

print("\nAll within-seed split overlap checks passed.")

# ------------------------------------------------------------
# 2. Cross-dataset path checks
#
# These should naturally be disjoint because they come from
# separate Kaggle dataset roots.
# ------------------------------------------------------------

assert path_set(source_base_df).isdisjoint(
    path_set(target_mild_base_df)
)

assert path_set(source_base_df).isdisjoint(
    path_set(target_hard_base_df)
)

assert path_set(source_base_df).isdisjoint(
    path_set(ood_test_df)
)

assert path_set(target_mild_base_df).isdisjoint(
    path_set(target_hard_base_df)
)

assert path_set(target_hard_base_df).isdisjoint(
    path_set(ood_test_df)
)

print("\nCross-dataset path checks passed.")

# ------------------------------------------------------------
# 3. Image decoding checker
# Same decoding logic used in the completed notebook.
# ------------------------------------------------------------
def scan_invalid_images(df, description, print_every=1000):
    bad_rows = []

    df_scan = df.reset_index(drop=True)

    print("\n" + "=" * 76)
    print(description)
    print("=" * 76)
    print("Files to check:", len(df_scan))

    for idx, row in df_scan.iterrows():
        fp = row["file_path"]

        try:
            raw = tf.io.read_file(fp)

            img = tf.image.decode_image(
                raw,
                channels=3,
                expand_animations=False
            )

            # Force TensorFlow to materialize/decode the image
            _ = tf.shape(img).numpy()

        except Exception as e:
            bad_rows.append({
                "row_index": idx,
                "file_path": fp,
                "canonical_class": row.get("canonical_class", None),
                "label": row.get("label", None),
                "error": str(e)[:500]
            })

        if (idx + 1) % print_every == 0:
            print(f"Checked {idx + 1}/{len(df_scan)}")

    bad_df = pd.DataFrame(bad_rows)

    print("\nBad files found:", len(bad_df))

    if len(bad_df) > 0:
        display(bad_df)

    return bad_df

# ------------------------------------------------------------
# 4. Scan hard-target train/test sets for all seeds
# ------------------------------------------------------------
invalid_image_reports = {}

for seed in SEEDS:
    s = all_seed_splits[seed]

    train_bad = scan_invalid_images(
        s["target_hard_train"],
        f"SEED {seed} — HARD TARGET TRAIN IMAGE CHECK"
    )

    test_bad = scan_invalid_images(
        s["target_hard_test"],
        f"SEED {seed} — HARD TARGET TEST IMAGE CHECK"
    )

    invalid_image_reports[seed] = {
        "train_bad": train_bad,
        "test_bad": test_bad
    }

    seed_dir = SPLIT_ROOT / f"seed_{seed}"

    train_bad.to_csv(
        seed_dir / f"invalid_hard_train_seed_{seed}.csv",
        index=False
    )

    test_bad.to_csv(
        seed_dir / f"invalid_hard_test_seed_{seed}.csv",
        index=False
    )

# ------------------------------------------------------------
# 5. Expected behavior
#
# Previous completed experiment found:
# - Seed 42 hard train: no bad files
# - Seed 123 hard train: no bad files
# - Seed 2024 hard train: one bad file
#
# Test sets should remain valid.
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("INVALID IMAGE SUMMARY")
print("=" * 76)

for seed in SEEDS:
    n_train_bad = len(
        invalid_image_reports[seed]["train_bad"]
    )

    n_test_bad = len(
        invalid_image_reports[seed]["test_bad"]
    )

    print(
        f"Seed {seed}: "
        f"train bad = {n_train_bad}, "
        f"test bad = {n_test_bad}"
    )

# ------------------------------------------------------------
# 6. Create ACTIVE hard-target training sets
#
# Do not alter original split CSVs.
# Only create cleaned copies where necessary.
# ------------------------------------------------------------

for seed in SEEDS:
    original_train = all_seed_splits[seed][
        "target_hard_train"
    ].copy()

    bad_df = invalid_image_reports[seed]["train_bad"]

    if len(bad_df) > 0:
        bad_paths = set(
            bad_df["file_path"].astype(str)
        )

        clean_train = original_train[
            ~original_train["file_path"]
            .astype(str)
            .isin(bad_paths)
        ].reset_index(drop=True)

    else:
        clean_train = original_train.reset_index(drop=True)

    all_seed_splits[seed][
        "target_hard_train_active"
    ] = clean_train

    seed_dir = SPLIT_ROOT / f"seed_{seed}"

    clean_train.to_csv(
        seed_dir / "target_hard_train_unlabeled_active.csv",
        index=False
    )

# ------------------------------------------------------------
# 7. Final active-size report
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("ACTIVE HARD-TARGET TRAINING SETS")
print("=" * 76)

for seed in SEEDS:
    active_df = all_seed_splits[seed][
        "target_hard_train_active"
    ]

    print(f"\nSeed {seed}")
    print("Active train size:", len(active_df))

    print(
        active_df["canonical_class"]
        .value_counts()
        .reindex(CLASS_NAMES)
        .to_string()
    )

# ------------------------------------------------------------
# 8. Final assertions
# ------------------------------------------------------------

for seed in SEEDS:

    # No invalid test image is acceptable.
    assert len(
        invalid_image_reports[seed]["test_bad"]
    ) == 0, (
        f"Seed {seed} contains invalid HARD TARGET TEST images. "
        "STOP before continuing."
    )

    # Cleaning must never create test overlap.
    assert_no_overlap(
        all_seed_splits[seed][
            "target_hard_train_active"
        ],
        all_seed_splits[seed][
            "target_hard_test"
        ],
        f"seed_{seed}_active_train",
        f"seed_{seed}_test"
    )

print("\n" + "=" * 76)
print("CELL 4 INTEGRITY CHECKS COMPLETED")
print("=" * 76)

print(
    "Important: original split files were preserved. "
    "Training will use target_hard_train_active."
)

SPLIT INTEGRITY CHECKS

Seed 42
  Source train/val/test overlap : 0
  Mild target train/test overlap: 0
  Hard target train/test overlap: 0

Seed 123
  Source train/val/test overlap : 0
  Mild target train/test overlap: 0
  Hard target train/test overlap: 0

Seed 2024
  Source train/val/test overlap : 0
  Mild target train/test overlap: 0
  Hard target train/test overlap: 0

All within-seed split overlap checks passed.

Cross-dataset path checks passed.

SEED 42 — HARD TARGET TRAIN IMAGE CHECK
Files to check: 7200


I0000 00:00:1789347904.115024      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789347904.117873      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Checked 1000/7200
Checked 2000/7200
Checked 3000/7200
Checked 4000/7200
Checked 5000/7200
Checked 6000/7200
Checked 7000/7200

Bad files found: 0

SEED 42 — HARD TARGET TEST IMAGE CHECK
Files to check: 1800
Checked 1000/1800

Bad files found: 0

SEED 123 — HARD TARGET TRAIN IMAGE CHECK
Files to check: 7200
Checked 1000/7200


2026-09-14 01:06:28.865378: E tensorflow/core/lib/jpeg/jpeg_mem.cc:329] Premature end of JPEG data. Stopped at line 558/750


Checked 2000/7200
Checked 3000/7200
Checked 4000/7200
Checked 5000/7200
Checked 6000/7200
Checked 7000/7200

Bad files found: 2


,row_index,file_path,canonical_class,label,error
0,1763,/kaggle/input/datasets/manhhoangvan/irishpotat...,Healthy,2,{{function_node __wrapped__DecodeImage_device_...
1,4741,/kaggle/input/datasets/manhhoangvan/irishpotat...,Early Blight,0,{{function_node __wrapped__DecodeImage_device_...



SEED 123 — HARD TARGET TEST IMAGE CHECK
Files to check: 1800
Checked 1000/1800

Bad files found: 0

SEED 2024 — HARD TARGET TRAIN IMAGE CHECK
Files to check: 7200
Checked 1000/7200
Checked 2000/7200
Checked 3000/7200
Checked 4000/7200
Checked 5000/7200
Checked 6000/7200
Checked 7000/7200

Bad files found: 0

SEED 2024 — HARD TARGET TEST IMAGE CHECK
Files to check: 1800
Checked 1000/1800

Bad files found: 0

INVALID IMAGE SUMMARY
Seed 42: train bad = 0, test bad = 0
Seed 123: train bad = 2, test bad = 0
Seed 2024: train bad = 0, test bad = 0

ACTIVE HARD-TARGET TRAINING SETS

Seed 42
Active train size: 7200
canonical_class
Early Blight    2400
Late Blight     2400
Healthy         2400

Seed 123
Active train size: 7198
canonical_class
Early Blight    2399
Late Blight     2400
Healthy         2399

Seed 2024
Active train size: 7200
canonical_class
Early Blight    2400
Late Blight     2400
Healthy         2400

CELL 4 INTEGRITY CHECKS COMPLETED
Important: original split files were preserv

In [5]:
# ============================================================
# CELL 5 — LOAD EXACT ORIGINAL EXPERIMENT MANIFESTS
#
# This REPLACES the reconstructed split manifests.
# From this point onward, external baselines use the exact
# train/test samples used in the completed main experiment.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Direct Kaggle input path
# The package is already extracted by Kaggle.
# ------------------------------------------------------------
PACKAGE_ROOT = Path(
    "/kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/"
    "TWD_paper_final_package_FIXED_20260905_234741"
)

assert PACKAGE_ROOT.exists(), (
    f"Package folder not found:\n{PACKAGE_ROOT}"
)

print("Package root found:")
print(PACKAGE_ROOT)

# ------------------------------------------------------------
# Find raw_result_files automatically
# ------------------------------------------------------------
raw_dirs = list(
    PACKAGE_ROOT.rglob("raw_result_files")
)

assert len(raw_dirs) >= 1, (
    "Could not locate raw_result_files inside the package.\n"
    "Run the directory diagnostic below and send me the output."
)

RAW_RESULTS = raw_dirs[0]

print("\nRaw result directory:")
print(RAW_RESULTS)

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------
def load_csv_checked(path):
    path = Path(path)

    assert path.exists(), f"Missing required file: {path}"

    df = pd.read_csv(path)

    required_cols = {
        "file_path",
        "canonical_class",
        "label"
    }

    missing = required_cols - set(df.columns)

    assert not missing, (
        f"{path.name} is missing columns: {missing}"
    )

    return df

# ------------------------------------------------------------
# SEED 42
# Root-level manifests correspond to original primary seed.
# ------------------------------------------------------------
seed42 = {
    "source_train": load_csv_checked(
        RAW_RESULTS / "source_train.csv"
    ),

    "source_val": load_csv_checked(
        RAW_RESULTS / "source_val.csv"
    ),

    "source_test": load_csv_checked(
        RAW_RESULTS / "source_test.csv"
    ),

    "target_hard_train": load_csv_checked(
        RAW_RESULTS / "target_hard_train_unlabeled.csv"
    ),

    "target_hard_test": load_csv_checked(
        RAW_RESULTS / "target_hard_test_labeled.csv"
    )
}

seed42["target_hard_train_active"] = (
    seed42["target_hard_train"].copy()
)

# ------------------------------------------------------------
# SEED 123
# ------------------------------------------------------------
SEED123_DIR = (
    RAW_RESULTS /
    "results_multiseed" /
    "seed_123"
)

seed123 = {
    "source_train": load_csv_checked(
        SEED123_DIR / "source_train.csv"
    ),

    "source_val": load_csv_checked(
        SEED123_DIR / "source_val.csv"
    ),

    "source_test": load_csv_checked(
        SEED123_DIR / "source_test.csv"
    ),

    "target_hard_train": load_csv_checked(
        SEED123_DIR / "target_hard_train_unlabeled.csv"
    ),

    "target_hard_test": load_csv_checked(
        SEED123_DIR / "target_hard_test_labeled.csv"
    )
}

seed123["target_hard_train_active"] = (
    seed123["target_hard_train"].copy()
)

# ------------------------------------------------------------
# SEED 2024
# ------------------------------------------------------------
SEED2024_DIR = (
    RAW_RESULTS /
    "results_multiseed" /
    "seed_2024"
)

seed2024 = {
    "source_train": load_csv_checked(
        SEED2024_DIR / "source_train.csv"
    ),

    "source_val": load_csv_checked(
        SEED2024_DIR / "source_val.csv"
    ),

    "source_test": load_csv_checked(
        SEED2024_DIR / "source_test.csv"
    ),

    "target_hard_train": load_csv_checked(
        SEED2024_DIR / "target_hard_train_unlabeled.csv"
    ),

    "target_hard_test": load_csv_checked(
        SEED2024_DIR / "target_hard_test_labeled.csv"
    ),

    "target_hard_train_active": load_csv_checked(
        SEED2024_DIR /
        "target_hard_train_unlabeled_cleaned.csv"
    )
}

# ------------------------------------------------------------
# Replace reconstructed split dictionary
# ------------------------------------------------------------
all_seed_splits = {
    42: seed42,
    123: seed123,
    2024: seed2024
}

print("\nExact original manifests loaded.")

Package root found:
/kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/TWD_paper_final_package_FIXED_20260905_234741

Raw result directory:
/kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/TWD_paper_final_package_FIXED_20260905_234741/raw_result_files

Exact original manifests loaded.


In [6]:
# ============================================================
# VERIFY EXACT ORIGINAL MANIFESTS
# ============================================================

EXPECTED_ACTIVE_TRAIN_SIZE = {
    42: 7200,
    123: 7200,
    2024: 7199
}

for seed in SEEDS:

    s = all_seed_splits[seed]

    print("\n" + "=" * 72)
    print(f"ORIGINAL MANIFEST CHECK — SEED {seed}")
    print("=" * 72)

    print("Source train:", len(s["source_train"]))
    print("Source val  :", len(s["source_val"]))
    print("Source test :", len(s["source_test"]))

    print(
        "Hard train original:",
        len(s["target_hard_train"])
    )

    print(
        "Hard train active  :",
        len(s["target_hard_train_active"])
    )

    print(
        "Hard test          :",
        len(s["target_hard_test"])
    )

    assert len(s["source_train"]) == 1506
    assert len(s["source_val"]) == 323
    assert len(s["source_test"]) == 323

    assert len(s["target_hard_train"]) == 7200
    assert len(s["target_hard_test"]) == 1800

    assert (
        len(s["target_hard_train_active"])
        == EXPECTED_ACTIVE_TRAIN_SIZE[seed]
    )

    print("\nActive hard-target train counts:")

    active_counts = (
        s["target_hard_train_active"]
        ["canonical_class"]
        .value_counts()
        .reindex(CLASS_NAMES)
        .fillna(0)
        .astype(int)
    )

    print(active_counts)

    print("\nHard-target test counts:")

    test_counts = (
        s["target_hard_test"]
        ["canonical_class"]
        .value_counts()
        .reindex(CLASS_NAMES)
        .fillna(0)
        .astype(int)
    )

    print(test_counts)

    assert (test_counts.values == 600).all()

    active_paths = set(
        s["target_hard_train_active"]
        ["file_path"]
        .astype(str)
    )

    test_paths = set(
        s["target_hard_test"]
        ["file_path"]
        .astype(str)
    )

    assert active_paths.isdisjoint(test_paths)

print("\n" + "=" * 72)
print("EXACT ORIGINAL SPLITS LOCKED SUCCESSFULLY")
print("=" * 72)


ORIGINAL MANIFEST CHECK — SEED 42
Source train: 1506
Source val  : 323
Source test : 323
Hard train original: 7200
Hard train active  : 7200
Hard test          : 1800

Active hard-target train counts:
canonical_class
Early Blight    2400
Late Blight     2400
Healthy         2400
Name: count, dtype: int64

Hard-target test counts:
canonical_class
Early Blight    600
Late Blight     600
Healthy         600
Name: count, dtype: int64

ORIGINAL MANIFEST CHECK — SEED 123
Source train: 1506
Source val  : 323
Source test : 323
Hard train original: 7200
Hard train active  : 7200
Hard test          : 1800

Active hard-target train counts:
canonical_class
Early Blight    2400
Late Blight     2400
Healthy         2400
Name: count, dtype: int64

Hard-target test counts:
canonical_class
Early Blight    600
Late Blight     600
Healthy         600
Name: count, dtype: int64

ORIGINAL MANIFEST CHECK — SEED 2024
Source train: 1506
Source val  : 323
Source test : 323
Hard train original: 7200
Hard train 

In [7]:
# ============================================================
# CELL 6 — SEARCH FOR ORIGINAL SOURCE MODEL CHECKPOINTS
#
# Best case:
#   Reuse the exact ResNet50 source models from the main study.
#
# Fallback:
#   If no .keras checkpoints are available, reconstruct and
#   retrain them using the exact original manifests + seeds.
# ============================================================

from pathlib import Path

print("=" * 76)
print("SEARCHING KAGGLE INPUTS FOR ORIGINAL KERAS MODELS")
print("=" * 76)

keras_files = list(
    Path("/kaggle/input").rglob("*.keras")
)

print(f"\nTotal .keras files found: {len(keras_files)}")

for p in keras_files:
    print(p)

# ------------------------------------------------------------
# Specifically look for source ResNet50 checkpoints
# ------------------------------------------------------------
source_model_candidates = [
    p for p in keras_files
    if "ResNet50_source_logit_model" in p.name
]

print("\n" + "=" * 76)
print("SOURCE RESNET50 CHECKPOINT CANDIDATES")
print("=" * 76)

if source_model_candidates:
    for p in source_model_candidates:
        print(p)
else:
    print("No original ResNet50 source-model .keras files found.")

print("\nSearch completed.")

SEARCHING KAGGLE INPUTS FOR ORIGINAL KERAS MODELS

Total .keras files found: 0

SOURCE RESNET50 CHECKPOINT CANDIDATES
No original ResNet50 source-model .keras files found.

Search completed.


In [8]:
# ============================================================
# CELL 7A — REMAP ORIGINAL SAVED PATHS TO CURRENT KAGGLE INPUTS
#
# IMPORTANT:
# We preserve the exact filenames/samples from the locked CSVs.
# Only the root directory is changed when necessary.
# ============================================================

from pathlib import Path
import os
import pandas as pd

# Current Kaggle dataset roots
SOURCE_ROOT = Path(
    "/kaggle/input/datasets/faysalmiah1721758/potato-dataset"
)

PLD_ROOT = Path(
    "/kaggle/input/datasets/shahadhossin567r7455/potato-leaf-disease-dataset"
)

IRISH_ROOT = Path(
    "/kaggle/input/datasets/manhhoangvan/irishpotato"
)

assert SOURCE_ROOT.exists()
assert PLD_ROOT.exists()
assert IRISH_ROOT.exists()

print("Current dataset roots found.")

# ------------------------------------------------------------
# Build filename -> current path indexes
# ------------------------------------------------------------
def build_filename_index(root):
    index = {}

    for p in root.rglob("*"):
        if p.is_file():
            index.setdefault(p.name, []).append(str(p))

    return index


print("\nBuilding file indexes...")

source_index = build_filename_index(SOURCE_ROOT)
irish_index = build_filename_index(IRISH_ROOT)
pld_index = build_filename_index(PLD_ROOT)

print("Indexes ready.")


# ------------------------------------------------------------
# Resolve one saved path
# ------------------------------------------------------------
def resolve_saved_path(old_path, dataset_type):

    old_path = str(old_path)

    # If original path still exists, keep it.
    if os.path.exists(old_path):
        return old_path

    filename = Path(old_path).name

    if dataset_type == "source":
        candidates = source_index.get(filename, [])

    elif dataset_type == "irish":
        candidates = irish_index.get(filename, [])

    elif dataset_type == "pld":
        candidates = pld_index.get(filename, [])

    else:
        raise ValueError(dataset_type)

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) == 0:
        return None

    # Multiple same filenames:
    # use class-folder clues from old path when possible.
    old_lower = old_path.lower()

    for candidate in candidates:
        c_lower = candidate.lower()

        if (
            ("early" in old_lower and "early" in c_lower)
            or ("late" in old_lower and "late" in c_lower)
            or ("healthy" in old_lower and "healthy" in c_lower)
        ):
            return candidate

    # Unsafe to guess.
    return None


# ------------------------------------------------------------
# Apply mapping to a DataFrame
# ------------------------------------------------------------
def remap_dataframe_paths(df, dataset_type):

    out = df.copy()

    out["original_file_path"] = out["file_path"]

    out["file_path"] = out["file_path"].apply(
        lambda x: resolve_saved_path(
            x,
            dataset_type
        )
    )

    missing = out["file_path"].isna().sum()

    return out, missing


# ------------------------------------------------------------
# Remap ALL locked manifests
# ------------------------------------------------------------
for seed in SEEDS:

    s = all_seed_splits[seed]

    for key in [
        "source_train",
        "source_val",
        "source_test"
    ]:
        s[key], missing = remap_dataframe_paths(
            s[key],
            "source"
        )

        assert missing == 0, (
            f"Seed {seed} {key}: "
            f"{missing} source files could not be resolved."
        )

    for key in [
        "target_hard_train",
        "target_hard_train_active",
        "target_hard_test"
    ]:
        s[key], missing = remap_dataframe_paths(
            s[key],
            "irish"
        )

        assert missing == 0, (
            f"Seed {seed} {key}: "
            f"{missing} Irish files could not be resolved."
        )

print("\nAll saved manifest paths successfully resolved.")

# ------------------------------------------------------------
# Final existence check
# ------------------------------------------------------------
for seed in SEEDS:

    s = all_seed_splits[seed]

    for key in [
        "source_train",
        "source_val",
        "source_test",
        "target_hard_train_active",
        "target_hard_test"
    ]:

        exists = s[key]["file_path"].apply(
            os.path.exists
        )

        assert exists.all(), (
            f"Broken paths remain in Seed {seed}, {key}"
        )

print("\n" + "=" * 72)
print("ORIGINAL SAMPLE IDENTITIES PRESERVED")
print("ALL CURRENT FILE PATHS VALID")
print("=" * 72)

Current dataset roots found.

Building file indexes...
Indexes ready.

All saved manifest paths successfully resolved.

ORIGINAL SAMPLE IDENTITIES PRESERVED
ALL CURRENT FILE PATHS VALID


In [13]:
# ============================================================
# CELL 8 — EXACT ORIGINAL SEED-42 SOURCE MODEL REPRODUCTION
#
# IMPORTANT:
# This follows the ACTUAL original notebook implementation:
#   map -> shuffle -> batch
#   ResNet50 built separately
#   base(x, training=False)
#   frozen BN during fine-tuning
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)
from sklearn.utils.class_weight import compute_class_weight

# ------------------------------------------------------------
# LOCKED SETTINGS
# ------------------------------------------------------------
RUN_SEED = 42

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 3
AUTOTUNE = tf.data.AUTOTUNE

CLASS_NAMES = [
    "Early Blight",
    "Late Blight",
    "Healthy"
]

# ------------------------------------------------------------
# RESET STATE EXACTLY BEFORE MODEL CREATION
# ------------------------------------------------------------
tf.keras.backend.clear_session()

random.seed(RUN_SEED)
np.random.seed(RUN_SEED)
tf.keras.utils.set_random_seed(RUN_SEED)

print("Exact source reproduction seed:", RUN_SEED)

# ------------------------------------------------------------
# EXACT ORIGINAL SAVED MANIFESTS
# ------------------------------------------------------------
source_train_df = all_seed_splits[RUN_SEED]["source_train"].copy()
source_val_df   = all_seed_splits[RUN_SEED]["source_val"].copy()
source_test_df  = all_seed_splits[RUN_SEED]["source_test"].copy()
hard_test_df    = all_seed_splits[RUN_SEED]["target_hard_test"].copy()

print("\nLocked manifests:")
print("Source train:", len(source_train_df))
print("Source val  :", len(source_val_df))
print("Source test :", len(source_test_df))
print("Hard test   :", len(hard_test_df))


# ============================================================
# ORIGINAL IMAGE PIPELINE
# ============================================================

def load_image_labeled_exact(file_path, label):
    img = tf.io.read_file(file_path)

    img = tf.image.decode_image(
        img,
        channels=3,
        expand_animations=False
    )

    img = tf.image.resize(
        img,
        [IMG_SIZE, IMG_SIZE]
    )

    img = tf.cast(img, tf.float32)

    return img, tf.cast(label, tf.int32)


def load_image_unlabeled_exact(file_path):
    img = tf.io.read_file(file_path)

    img = tf.image.decode_image(
        img,
        channels=3,
        expand_animations=False
    )

    img = tf.image.resize(
        img,
        [IMG_SIZE, IMG_SIZE]
    )

    img = tf.cast(img, tf.float32)

    return img


def make_labeled_ds_exact(df, shuffle=False):

    file_paths = df["file_path"].values
    labels = df["label"].values

    ds = tf.data.Dataset.from_tensor_slices(
        (file_paths, labels)
    )

    # ORIGINAL ORDER:
    # map first
    ds = ds.map(
        load_image_labeled_exact,
        num_parallel_calls=AUTOTUNE
    )

    # shuffle second
    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(df),
            seed=42,
            reshuffle_each_iteration=True
        )

    # batch third
    ds = ds.batch(BATCH_SIZE)

    # prefetch last
    ds = ds.prefetch(AUTOTUNE)

    return ds


def make_unlabeled_ds_exact(df):

    file_paths = df["file_path"].values

    ds = tf.data.Dataset.from_tensor_slices(
        file_paths
    )

    ds = ds.map(
        load_image_unlabeled_exact,
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)

    return ds


source_train_ds = make_labeled_ds_exact(
    source_train_df,
    shuffle=True
)

source_val_ds = make_labeled_ds_exact(
    source_val_df,
    shuffle=False
)

source_test_ds = make_labeled_ds_exact(
    source_test_df,
    shuffle=False
)

hard_test_ds = make_labeled_ds_exact(
    hard_test_df,
    shuffle=False
)


# ============================================================
# ORIGINAL SOURCE CLASS WEIGHTS
# ============================================================

classes = np.array([0, 1, 2])

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=source_train_df["label"].values
)

class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(
        classes,
        class_weights_array
    )
}

print("\nClass weights:")
print(class_weights)


# ============================================================
# EXACT ORIGINAL RESNET50 LOGIT MODEL
# ============================================================

def build_resnet50_exact():

    inputs = layers.Input(
        shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    # preprocessing occurs BEFORE the separately-built backbone
    x = tf.keras.applications.resnet50.preprocess_input(
        inputs
    )

    # IMPORTANT:
    # Do NOT use input_tensor=x here.
    base = tf.keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    base.trainable = False

    # IMPORTANT:
    # Exact original notebook behavior.
    x = base(
        x,
        training=False
    )

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dropout(
        0.30
    )(x)

    logits = layers.Dense(
        NUM_CLASSES,
        activation=None,
        name="logits"
    )(x)

    model = models.Model(
        inputs,
        logits,
        name="ResNet50_logit_model"
    )

    return model, base


model, base = build_resnet50_exact()

print("\nModel created.")


# ============================================================
# STAGE 1 — EXACT ORIGINAL HEAD TRAINING
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),

    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),

    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

callbacks_head = [

    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]

print("\n" + "=" * 72)
print("STAGE 1 — EXACT ORIGINAL HEAD TRAINING")
print("=" * 72)

history_head_exact = model.fit(
    source_train_ds,
    validation_data=source_val_ds,
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks_head,
    verbose=1
)


# ============================================================
# STAGE 2 — EXACT ORIGINAL BACKBONE FINE-TUNING
# ============================================================

base.trainable = True

# Freeze all BatchNorm layers.
for layer in base.layers:
    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = False


model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),

    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),

    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

callbacks_ft = [

    EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7
    )
]

print("\n" + "=" * 72)
print("STAGE 2 — EXACT ORIGINAL FINE-TUNING")
print("=" * 72)

history_ft_exact = model.fit(
    source_train_ds,
    validation_data=source_val_ds,
    epochs=30,
    class_weight=class_weights,
    callbacks=callbacks_ft,
    verbose=1
)

print("\nExact Seed-42 reproduction training completed.")

Exact source reproduction seed: 42

Locked manifests:
Source train: 1506
Source val  : 323
Source test : 323
Hard test   : 1800

Class weights:
{0: 0.7171428571428572, 1: 0.7171428571428572, 2: 4.735849056603773}

Model created.

STAGE 1 — EXACT ORIGINAL HEAD TRAINING
Epoch 1/20
48/48 ━━━━━━━━━━━━━━━━━━━━ 25s 292ms/step - accuracy: 0.7729 - loss: 0.5562 - val_accuracy: 0.9412 - val_loss: 0.1664 - learning_rate: 0.0010
Epoch 2/20
48/48 ━━━━━━━━━━━━━━━━━━━━ 6s 101ms/step - accuracy: 0.9449 - loss: 0.1626 - val_accuracy: 0.9845 - val_loss: 0.0723 - learning_rate: 0.0010
Epoch 3/20
48/48 ━━━━━━━━━━━━━━━━━━━━ 6s 98ms/step - accuracy: 0.9502 - loss: 0.1428 - val_accuracy: 0.9659 - val_loss: 0.1009 - learning_rate: 0.0010
Epoch 4/20
48/48 ━━━━━━━━━━━━━━━━━━━━ 6s 101ms/step - accuracy: 0.9416 - loss: 0.1240 - val_accuracy: 0.9845 - val_loss: 0.0469 - learning_rate: 0.0010
Epoch 5/20
48/48 ━━━━━━━━━━━━━━━━━━━━ 6s 102ms/step - accuracy: 0.9761 - loss: 0.0719 - val_accuracy: 0.9969 - val_loss: 0.

In [14]:
# ============================================================
# CELL 9 — EXACT REPRODUCTION EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix
)

def expected_calibration_error_exact(
    y_true,
    probs,
    n_bins=15
):
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = predictions == y_true

    ece = 0.0

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    for i in range(n_bins):

        left = bin_edges[i]
        right = bin_edges[i + 1]

        if i == n_bins - 1:
            mask = (
                (confidences >= left)
                & (confidences <= right)
            )
        else:
            mask = (
                (confidences >= left)
                & (confidences < right)
            )

        if np.any(mask):
            ece += (
                np.mean(mask)
                * abs(
                    np.mean(accuracies[mask])
                    - np.mean(confidences[mask])
                )
            )

    return float(ece)


def brier_exact(y_true, probs):
    y_onehot = np.eye(NUM_CLASSES)[y_true]

    return float(
        np.mean(
            np.sum(
                (probs - y_onehot) ** 2,
                axis=1
            )
        )
    )


def nll_exact(y_true, probs, eps=1e-12):

    probs = np.clip(
        probs,
        eps,
        1.0
    )

    return float(
        -np.mean(
            np.log(
                probs[
                    np.arange(len(y_true)),
                    y_true
                ]
            )
        )
    )


def evaluate_exact(model, df, name):

    infer_ds = make_unlabeled_ds_exact(df)

    logits = model.predict(
        infer_ds,
        verbose=1
    )

    probs = tf.nn.softmax(
        logits,
        axis=1
    ).numpy()

    y_true = df["label"].values.astype(int)

    y_pred = np.argmax(
        probs,
        axis=1
    )

    result = {
        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro"
        ),

        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted"
        ),

        "ece": expected_calibration_error_exact(
            y_true,
            probs
        ),

        "brier": brier_exact(
            y_true,
            probs
        ),

        "nll": nll_exact(
            y_true,
            probs
        )
    }

    print("\n" + "=" * 72)
    print(name)
    print("=" * 72)

    for k, v in result.items():
        print(f"{k:12s}: {v:.6f}")

    print("\nConfusion matrix:")
    print(
        confusion_matrix(
            y_true,
            y_pred
        )
    )

    return result, probs, y_pred


source_exact, _, _ = evaluate_exact(
    model,
    source_test_df,
    "SOURCE TEST"
)

hard_exact, hard_probs_exact, hard_preds_exact = evaluate_exact(
    model,
    hard_test_df,
    "HARD TARGET"
)


# ============================================================
# COMPARE TO ORIGINAL COMPLETED EXPERIMENT
# ============================================================

ORIGINAL = {
    "accuracy": 0.813333,
    "macro_f1": 0.805141,
    "ece": 0.131565,
    "brier": 0.315410,
    "nll": 0.990472
}

print("\n" + "=" * 72)
print("EXACT REPRODUCTION VS ORIGINAL SEED 42")
print("=" * 72)

for metric, original_value in ORIGINAL.items():

    reproduced = hard_exact[metric]

    print(
        f"{metric:12s} | "
        f"original={original_value:.6f} | "
        f"reproduced={reproduced:.6f} | "
        f"diff={reproduced-original_value:+.6f}"
    )

print("\nAccuracy difference (percentage points):")
print(
    (hard_exact["accuracy"] - ORIGINAL["accuracy"])
    * 100
)

11/11 ━━━━━━━━━━━━━━━━━━━━ 8s 449ms/step

SOURCE TEST
accuracy    : 1.000000
macro_f1    : 1.000000
weighted_f1 : 1.000000
ece         : 0.001519
brier       : 0.000486
nll         : 0.001664

Confusion matrix:
[[150   0   0]
 [  0 150   0]
 [  0   0  23]]
57/57 ━━━━━━━━━━━━━━━━━━━━ 7s 120ms/step

HARD TARGET
accuracy    : 0.840000
macro_f1    : 0.834886
weighted_f1 : 0.834886
ece         : 0.106917
brier       : 0.261456
nll         : 0.730451

Confusion matrix:
[[583  17   0]
 [ 36 562   2]
 [189  44 367]]

EXACT REPRODUCTION VS ORIGINAL SEED 42
accuracy     | original=0.813333 | reproduced=0.840000 | diff=+0.026667
macro_f1     | original=0.805141 | reproduced=0.834886 | diff=+0.029745
ece          | original=0.131565 | reproduced=0.106917 | diff=-0.024648
brier        | original=0.315410 | reproduced=0.261456 | diff=-0.053954
nll          | original=0.990472 | reproduced=0.730451 | diff=-0.260021

Accuracy difference (percentage points):
2.6666999999999996


In [15]:
# ============================================================
# CELL 10 — LOCK REPRODUCED SEED-42 CHECKPOINT
# + BUILD CONTROLLED THREE-WAY PARTITION
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

# ------------------------------------------------------------
# Controlled-comparison output directory
# ------------------------------------------------------------
CONTROL_DIR = Path(
    "/kaggle/working/external_sfuda_baseline/"
    "controlled_seed42"
)

CONTROL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SOURCE_MODEL_PATH = (
    CONTROL_DIR /
    "ResNet50_reproduced_source_seed42.keras"
)

model.save(SOURCE_MODEL_PATH)

print("Saved reproduced source checkpoint:")
print(SOURCE_MODEL_PATH)

# ------------------------------------------------------------
# LOCKED proposed-method configuration
# ------------------------------------------------------------
CONF_ACCEPT = 0.9999
CONF_REJECT = 0.6667
MARGIN_ACCEPT = 0.0

ACCEPT_WEIGHT = 1.0
DEFER_WEIGHT = 0.3
T_DEFER = 2.0

CLASS_WEIGHT_CAP = 3.0

RUN_SEED = 42

# ------------------------------------------------------------
# Exact active target-training split
# ------------------------------------------------------------
target_train_df = (
    all_seed_splits[RUN_SEED]
    ["target_hard_train_active"]
    .copy()
    .reset_index(drop=True)
)

print("\nTarget training samples:",
      len(target_train_df))

# ------------------------------------------------------------
# Prediction dataset
# ------------------------------------------------------------
target_train_infer_ds = make_unlabeled_ds_exact(
    target_train_df
)

# ------------------------------------------------------------
# Fixed SOURCE TEACHER predictions
# ------------------------------------------------------------
teacher_logits = model.predict(
    target_train_infer_ds,
    verbose=1
)

teacher_probs = tf.nn.softmax(
    teacher_logits,
    axis=1
).numpy()

teacher_pred = np.argmax(
    teacher_probs,
    axis=1
)

teacher_conf = np.max(
    teacher_probs,
    axis=1
)

# ------------------------------------------------------------
# Logit margin
# Original diagnostic definition:
# top logit minus second-highest logit
# ------------------------------------------------------------
sorted_logits = np.sort(
    teacher_logits,
    axis=1
)

teacher_margin = (
    sorted_logits[:, -1]
    - sorted_logits[:, -2]
)

# ------------------------------------------------------------
# Store teacher outputs
# ------------------------------------------------------------
pred_df = target_train_df.copy()

pred_df["pred_label"] = teacher_pred
pred_df["confidence"] = teacher_conf
pred_df["logit_margin"] = teacher_margin

for c in range(NUM_CLASSES):
    pred_df[f"logit_{CLASS_NAMES[c]}"] = (
        teacher_logits[:, c]
    )

# ------------------------------------------------------------
# Exact three-way decision rule
# ------------------------------------------------------------
def assign_region(row):

    if (
        row["confidence"] >= CONF_ACCEPT
        and
        row["logit_margin"] >= MARGIN_ACCEPT
    ):
        return "Accept"

    elif row["confidence"] < CONF_REJECT:
        return "Reject"

    else:
        return "Defer"


pred_df["region"] = pred_df.apply(
    assign_region,
    axis=1
)

print("\nThree-way region counts:")
print(
    pred_df["region"]
    .value_counts()
)

print("\nThree-way region coverage:")
print(
    pred_df["region"]
    .value_counts(normalize=True)
    .sort_index()
)

# ------------------------------------------------------------
# Only Accept + Defer contribute to adaptation
# ------------------------------------------------------------
target_used_df = (
    pred_df[
        pred_df["region"].isin(
            ["Accept", "Defer"]
        )
    ]
    .copy()
    .reset_index(drop=True)
)

target_used_df["pseudo_label"] = (
    target_used_df["pred_label"]
    .astype(int)
)

print("\nSamples used for adaptation:")
print(len(target_used_df))

print("\nTraining-region counts:")
print(
    target_used_df["region"]
    .value_counts()
)

# ------------------------------------------------------------
# Save FIXED partition
# ------------------------------------------------------------
pred_df.to_csv(
    CONTROL_DIR /
    "seed42_source_teacher_threeway_partition.csv",
    index=False
)

print("\nSaved fixed teacher partition.")

Saved reproduced source checkpoint:
/kaggle/working/external_sfuda_baseline/controlled_seed42/ResNet50_reproduced_source_seed42.keras

Target training samples: 7200
225/225 ━━━━━━━━━━━━━━━━━━━━ 20s 86ms/step

Three-way region counts:
region
Defer     3898
Accept    2869
Reject     433
Name: count, dtype: int64

Three-way region coverage:
region
Accept    0.398472
Defer     0.541389
Reject    0.060139
Name: proportion, dtype: float64

Samples used for adaptation:
6767

Training-region counts:
region
Defer     3898
Accept    2869
Name: count, dtype: int64

Saved fixed teacher partition.


In [16]:
# ============================================================
# CELL 11 — CONTROLLED SEED-42
# FULL THREE-WAY CLASS-AWARE ADAPTATION
#
# SAME reproduced source checkpoint that the external
# baseline will later receive.
# ============================================================

# ------------------------------------------------------------
# Class-aware weights from ACCEPT pseudo-label distribution
# ------------------------------------------------------------
accept_df = target_used_df[
    target_used_df["region"] == "Accept"
].copy()

accept_counts = (
    accept_df["pseudo_label"]
    .value_counts()
    .reindex(
        [0, 1, 2],
        fill_value=1
    )
)

reference_count = np.median(
    accept_counts.values
)

class_balance_weights = {}

for cls_id, count in accept_counts.items():

    raw_weight = np.sqrt(
        reference_count / count
    )

    clipped_weight = np.clip(
        raw_weight,
        0.5,
        CLASS_WEIGHT_CAP
    )

    class_balance_weights[
        int(cls_id)
    ] = float(clipped_weight)


print("Accepted pseudo-label counts:")
print(accept_counts)

print("\nClass-aware weights:")
print(class_balance_weights)


# ------------------------------------------------------------
# Teacher soft targets
# ------------------------------------------------------------
def softmax_np(
    x,
    temperature=1.0
):

    x = x / temperature

    x = (
        x
        - np.max(
            x,
            axis=1,
            keepdims=True
        )
    )

    exp_x = np.exp(x)

    return (
        exp_x
        / np.sum(
            exp_x,
            axis=1,
            keepdims=True
        )
    )


logit_cols = [
    f"logit_{cls}"
    for cls in CLASS_NAMES
]

used_logits = (
    target_used_df[
        logit_cols
    ]
    .values
    .astype("float32")
)

teacher_probs_soft = softmax_np(
    used_logits,
    temperature=T_DEFER
)

pseudo_labels = (
    target_used_df[
        "pseudo_label"
    ]
    .values
    .astype(int)
)

onehot_pseudo = np.eye(
    NUM_CLASSES
)[pseudo_labels].astype("float32")

is_accept = (
    target_used_df["region"].values
    == "Accept"
)

is_defer = (
    target_used_df["region"].values
    == "Defer"
)

# ------------------------------------------------------------
# Accept = hard pseudo-label
# Defer  = softened fixed teacher probability
# ------------------------------------------------------------
soft_targets = np.zeros(
    (
        len(target_used_df),
        NUM_CLASSES
    ),
    dtype="float32"
)

soft_targets[is_accept] = (
    onehot_pseudo[is_accept]
)

soft_targets[is_defer] = (
    teacher_probs_soft[is_defer]
)

# ------------------------------------------------------------
# Sample weights
# ------------------------------------------------------------
sample_weights = np.zeros(
    len(target_used_df),
    dtype="float32"
)

for i, pseudo_label in enumerate(
    pseudo_labels
):

    class_weight = (
        class_balance_weights[
            int(pseudo_label)
        ]
    )

    if is_accept[i]:

        sample_weights[i] = (
            ACCEPT_WEIGHT
            * class_weight
        )

    elif is_defer[i]:

        sample_weights[i] = (
            DEFER_WEIGHT
            * class_weight
        )

# Same normalization as original experiment
sample_weights = (
    sample_weights
    / np.mean(sample_weights)
)

print("\nSample-weight mean:",
      np.mean(sample_weights))


# ============================================================
# DATASET
# ============================================================

def load_soft_sample(
    file_path,
    soft_label,
    sample_weight
):

    img = tf.io.read_file(
        file_path
    )

    img = tf.image.decode_image(
        img,
        channels=3,
        expand_animations=False
    )

    img = tf.image.resize(
        img,
        [IMG_SIZE, IMG_SIZE]
    )

    img = tf.cast(
        img,
        tf.float32
    )

    return (
        img,
        tf.cast(
            soft_label,
            tf.float32
        ),
        tf.cast(
            sample_weight,
            tf.float32
        )
    )


def make_soft_ds(
    df,
    targets,
    weights
):

    ds = (
        tf.data.Dataset
        .from_tensor_slices(
            (
                df["file_path"].values,
                targets,
                weights
            )
        )
    )

    ds = ds.map(
        load_soft_sample,
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.shuffle(
        len(df),
        seed=RUN_SEED,
        reshuffle_each_iteration=True
    )

    ds = ds.batch(
        BATCH_SIZE
    )

    ds = ds.prefetch(
        AUTOTUNE
    )

    return ds


threeway_train_ds = make_soft_ds(
    target_used_df,
    soft_targets,
    sample_weights
)


# ============================================================
# LOAD SAME REPRODUCED SOURCE CHECKPOINT
# ============================================================

threeway_model = (
    tf.keras.models.load_model(
        SOURCE_MODEL_PATH
    )
)

threeway_model.trainable = True


def freeze_batchnorm_recursive(
    model
):

    for layer in model.layers:

        if isinstance(
            layer,
            tf.keras.layers.BatchNormalization
        ):
            layer.trainable = False

        elif hasattr(
            layer,
            "layers"
        ):
            freeze_batchnorm_recursive(
                layer
            )


freeze_batchnorm_recursive(
    threeway_model
)

threeway_model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=5e-6
    ),

    loss=tf.keras.losses.CategoricalCrossentropy(
        from_logits=True
    ),

    metrics=[
        tf.keras.metrics.CategoricalAccuracy(
            name="accuracy"
        )
    ]
)


print("\n" + "=" * 72)
print("CONTROLLED SEED-42 THREE-WAY ADAPTATION")
print("=" * 72)

history_threeway = (
    threeway_model.fit(
        threeway_train_ds,
        epochs=5,
        verbose=1
    )
)

THREEWAY_MODEL_PATH = (
    CONTROL_DIR /
    "ResNet50_threeway_classaware_controlled_seed42.keras"
)

threeway_model.save(
    THREEWAY_MODEL_PATH
)

print("\nSaved:")
print(THREEWAY_MODEL_PATH)

Accepted pseudo-label counts:
pseudo_label
0    1312
1    1514
2      43
Name: count, dtype: int64

Class-aware weights:
{0: 1.0, 1: 0.9309020355202793, 2: 3.0}

Sample-weight mean: 1.0

CONTROLLED SEED-42 THREE-WAY ADAPTATION
Epoch 1/5
212/212 ━━━━━━━━━━━━━━━━━━━━ 114s 352ms/step - accuracy: 0.9586 - loss: 0.2250
Epoch 2/5
212/212 ━━━━━━━━━━━━━━━━━━━━ 70s 276ms/step - accuracy: 0.9694 - loss: 0.2079
Epoch 3/5
212/212 ━━━━━━━━━━━━━━━━━━━━ 70s 276ms/step - accuracy: 0.9780 - loss: 0.2029
Epoch 4/5
212/212 ━━━━━━━━━━━━━━━━━━━━ 70s 276ms/step - accuracy: 0.9799 - loss: 0.2008
Epoch 5/5
212/212 ━━━━━━━━━━━━━━━━━━━━ 69s 276ms/step - accuracy: 0.9814 - loss: 0.1997

Saved:
/kaggle/working/external_sfuda_baseline/controlled_seed42/ResNet50_threeway_classaware_controlled_seed42.keras


In [17]:
# ============================================================
# CELL 12 — CONTROLLED SEED-42 EVALUATION
# SAME SOURCE CHECKPOINT FOR FAIR COMPARISON
# ============================================================

import pandas as pd
import tensorflow as tf

# ------------------------------------------------------------
# Reload both checkpoints fresh
# ------------------------------------------------------------
source_control_model = tf.keras.models.load_model(
    SOURCE_MODEL_PATH
)

proposed_control_model = tf.keras.models.load_model(
    THREEWAY_MODEL_PATH
)

# ------------------------------------------------------------
# Evaluate SOURCE ONLY
# ------------------------------------------------------------
source_control_metrics, _, _ = evaluate_exact(
    source_control_model,
    hard_test_df,
    "CONTROLLED SOURCE-ONLY — HARD TARGET"
)

# ------------------------------------------------------------
# Evaluate PROPOSED METHOD
# ------------------------------------------------------------
proposed_control_metrics, _, _ = evaluate_exact(
    proposed_control_model,
    hard_test_df,
    "CONTROLLED THREE-WAY CLASS-AWARE — HARD TARGET"
)

# ------------------------------------------------------------
# Build comparison table
# ------------------------------------------------------------
comparison_controlled = pd.DataFrame([
    {
        "Method": "SourceOnly",
        "Accuracy": source_control_metrics["accuracy"],
        "Macro_F1": source_control_metrics["macro_f1"],
        "ECE": source_control_metrics["ece"],
        "Brier": source_control_metrics["brier"],
        "NLL": source_control_metrics["nll"],
    },
    {
        "Method": "ThreeWayClassAware",
        "Accuracy": proposed_control_metrics["accuracy"],
        "Macro_F1": proposed_control_metrics["macro_f1"],
        "ECE": proposed_control_metrics["ece"],
        "Brier": proposed_control_metrics["brier"],
        "NLL": proposed_control_metrics["nll"],
    }
])

print("\n" + "=" * 78)
print("CONTROLLED SEED-42: SOURCE ONLY VS PROPOSED")
print("=" * 78)

display(
    comparison_controlled.round(6)
)

# ------------------------------------------------------------
# Improvement calculations
# ------------------------------------------------------------
acc_gain_pp = (
    proposed_control_metrics["accuracy"]
    - source_control_metrics["accuracy"]
) * 100

f1_gain_pp = (
    proposed_control_metrics["macro_f1"]
    - source_control_metrics["macro_f1"]
) * 100

ece_reduction = (
    source_control_metrics["ece"]
    - proposed_control_metrics["ece"]
)

brier_reduction = (
    source_control_metrics["brier"]
    - proposed_control_metrics["brier"]
)

nll_reduction = (
    source_control_metrics["nll"]
    - proposed_control_metrics["nll"]
)

print("\nChanges relative to the SAME source checkpoint:")
print(f"Accuracy gain : {acc_gain_pp:+.3f} percentage points")
print(f"Macro-F1 gain: {f1_gain_pp:+.3f} percentage points")
print(f"ECE reduction: {ece_reduction:+.6f}")
print(f"Brier reduction: {brier_reduction:+.6f}")
print(f"NLL reduction: {nll_reduction:+.6f}")

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
comparison_path = (
    CONTROL_DIR /
    "controlled_seed42_source_vs_threeway.csv"
)

comparison_controlled.to_csv(
    comparison_path,
    index=False
)

print("\nSaved comparison:")
print(comparison_path)

57/57 ━━━━━━━━━━━━━━━━━━━━ 12s 143ms/step

CONTROLLED SOURCE-ONLY — HARD TARGET
accuracy    : 0.840000
macro_f1    : 0.834886
weighted_f1 : 0.834886
ece         : 0.106917
brier       : 0.261456
nll         : 0.730451

Confusion matrix:
[[583  17   0]
 [ 36 562   2]
 [189  44 367]]
57/57 ━━━━━━━━━━━━━━━━━━━━ 12s 145ms/step

CONTROLLED THREE-WAY CLASS-AWARE — HARD TARGET
accuracy    : 0.843333
macro_f1    : 0.838457
weighted_f1 : 0.838457
ece         : 0.046619
brier       : 0.233041
nll         : 0.459743

Confusion matrix:
[[587  13   0]
 [ 35 563   2]
 [194  38 368]]

CONTROLLED SEED-42: SOURCE ONLY VS PROPOSED


,Method,Accuracy,Macro_F1,ECE,Brier,NLL
0,SourceOnly,0.840000,0.834886,0.106917,0.261456,0.730451
1,ThreeWayClassAware,0.843333,0.838457,0.046619,0.233041,0.459743



Changes relative to the SAME source checkpoint:
Accuracy gain : +0.333 percentage points
Macro-F1 gain: +0.357 percentage points
ECE reduction: +0.060298
Brier reduction: +0.028415
NLL reduction: +0.270708

Saved comparison:
/kaggle/working/external_sfuda_baseline/controlled_seed42/controlled_seed42_source_vs_threeway.csv


In [18]:
# ============================================================
# CELL 13 — DOWNLOAD OFFICIAL UPA IMPLEMENTATION
# Neurocomputing 2024
# ============================================================

import os
import subprocess
from pathlib import Path

UPA_DIR = Path("/kaggle/working/UPA_official")

if not UPA_DIR.exists():

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/chenxi52/UPA.git",
            str(UPA_DIR)
        ],
        check=True
    )

else:
    print("UPA repository already exists.")

print("\nUPA repository:")
print(UPA_DIR)

print("\nTop-level files:")
for p in sorted(UPA_DIR.iterdir()):
    print(" -", p.name)

print("\n" + "=" * 72)
print("IMPORTANT TRAINING FILES")
print("=" * 72)

for filename in [
    "upa_v2_amp.py",
    "train_source.py",
    "README.md"
]:

    path = UPA_DIR / filename

    print(
        filename,
        "->",
        "FOUND" if path.exists() else "MISSING"
    )

Cloning into '/kaggle/working/UPA_official'...



UPA repository:
/kaggle/working/UPA_official

Top-level files:
 - .git
 - .gitignore
 - README.md
 - figs
 - loaders
 - model
 - run_domainnet126.sh
 - run_office-home.sh
 - run_office.sh
 - run_source.sh
 - run_visda.sh
 - train_source.py
 - trainer
 - upa_v2_amp.py
 - utils

IMPORTANT TRAINING FILES
upa_v2_amp.py -> FOUND
train_source.py -> FOUND
README.md -> FOUND


In [19]:
# ============================================================
# CELL 14 — INSPECT OFFICIAL UPA STRUCTURE
# NO TRAINING YET
# ============================================================

from pathlib import Path

files_to_inspect = [
    UPA_DIR / "upa_v2_amp.py",
    UPA_DIR / "README.md"
]

for file_path in files_to_inspect:

    print("\n")
    print("=" * 90)
    print(file_path.name)
    print("=" * 90)

    if not file_path.exists():
        print("FILE NOT FOUND")
        continue

    text = file_path.read_text(
        errors="ignore"
    )

    lines = text.splitlines()

    print("Total lines:", len(lines))

    # Show imports and argument definitions first.
    interesting = []

    keywords = [
        "argparse",
        "add_argument",
        "ResNet",
        "resnet",
        "optimizer",
        "lr",
        "epoch",
        "neighbor",
        "contrast",
        "pseudo",
        "uncert",
        "threshold",
        "SHOT",
        "classifier",
        "feature",
        "bottleneck"
    ]

    for i, line in enumerate(lines, start=1):

        if any(
            keyword.lower() in line.lower()
            for keyword in keywords
        ):
            interesting.append(
                (i, line)
            )

    print("\nRelevant lines:\n")

    for i, line in interesting[:250]:
        print(
            f"{i:04d}: {line}"
        )



upa_v2_amp.py
Total lines: 164

Relevant lines:

0003: import argparse
0016:     parser = argparse.ArgumentParser(description='SHOT')
0017:     parser.add_argument('--s', type=int, default=0, help="source")
0018:     parser.add_argument('--t', type=int, default=1, help="target")
0019:     parser.add_argument('--max_epoch', type=int, default=15, help="max iterations")
0020:     parser.add_argument('--batch_size', type=int, default=64, help="batch_size")
0021:     parser.add_argument('--worker', type=int, default=4, help="number of workers")
0022:     parser.add_argument('--dset', type=str, default='office-home',
0024:     parser.add_argument('--lr', type=float, default=1e-2, help="learning rate")
0025:     parser.add_argument('--seed', type=int, default=2020, help="random seed")
0026:     parser.add_argument('--output', type=str, default='san')
0027:     parser.add_argument('--output_src', type=str, default='res/ckps/source')
0028:     parser.add_argument('--da', type=str, default='ud

In [20]:
# ============================================================
# CELL 15 — UPA ENVIRONMENT / IMPORT CHECK
# NO ADAPTATION
# ============================================================

import sys
import torch
import torchvision

print("Python      :", sys.version)
print("PyTorch     :", torch.__version__)
print("Torchvision :", torchvision.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print("\nTensorFlow checkpoint retained:")
print(SOURCE_MODEL_PATH)

print("\nLocked target training set:")
print(len(target_train_df))

print("\nLocked hard target test set:")
print(len(hard_test_df))

print("\n" + "=" * 72)
print("UPA PREPARATION COMPLETE — NO UPA TRAINING PERFORMED")
print("=" * 72)

Python      : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch     : 2.10.0+cu128
Torchvision : 0.25.0+cu128

CUDA available: True
GPU: Tesla T4

TensorFlow checkpoint retained:
/kaggle/working/external_sfuda_baseline/controlled_seed42/ResNet50_reproduced_source_seed42.keras

Locked target training set:
7200

Locked hard target test set:
1800

UPA PREPARATION COMPLETE — NO UPA TRAINING PERFORMED


In [21]:
# ============================================================
# CELL 16 — INSPECT UPA'S UNCERTAINTY / PAIR-SELECTION LOGIC
# NO TRAINING
# ============================================================

from pathlib import Path

files_to_check = [
    UPA_DIR / "utils" / "utils_noise.py",
    UPA_DIR / "utils" / "loss.py",
    UPA_DIR / "loaders" / "target_loader.py",
    UPA_DIR / "loaders" / "data_list.py",
]

keywords = [
    "pair_selection",
    "selected_examples",
    "selected_pairs",
    "knn",
    "uncert",
    "entropy",
    "prob",
    "cos",
    "weight",
    "balance",
    "Pseudo_dataset",
]

for path in files_to_check:

    print("\n" + "=" * 90)
    print(path)
    print("=" * 90)

    if not path.exists():
        print("MISSING")
        continue

    lines = path.read_text(
        errors="ignore"
    ).splitlines()

    print("Total lines:", len(lines))

    matches = []

    for i, line in enumerate(
        lines,
        start=1
    ):

        if any(
            k.lower() in line.lower()
            for k in keywords
        ):
            matches.append((i, line))

    for i, line in matches[:350]:
        print(f"{i:04d}: {line}")


/kaggle/working/UPA_official/utils/utils_noise.py
Total lines: 96
0007: def pair_selection_v1(k_val, test_loader, labels, class_num, cos_t, knn_times, train_features, balance_class=True,
0010:     k_val:  neighbors number of knn
0020:         for i in range(knn_times):
0021:             print(f'starting the {i+1}st knn....')
0041:                 yd_transform = torch.exp(yd.clone().div_(cos_t))  
0042:                 probs_corrected = torch.sum(torch.mul(retrieval_one_hot_train.view(batch_size, -1, class_num),
0044:                 probs_norm = probs_corrected / torch.sum(probs_corrected, dim=1)[:, None]
0046:                 prob_temp = probs_norm[torch.arange(0, batch_size), labels[index]]
0047:                 prob_temp[prob_temp <= 1e-4] = 1e-4
0048:                 prob_temp[prob_temp > (1 - 1e-4)] = 1 - 1e-4
0049:                 discrepancy_measure[index] = -torch.log(prob_temp)
0052:                 sorted_pro, predictions_corrected = probs_norm.sort(1, True)
0055:           

In [22]:
# ============================================================
# CELL 17 — SHOW COMPLETE pair_selection_v1 FUNCTION
# ============================================================

import inspect
import sys

sys.path.insert(
    0,
    str(UPA_DIR)
)

from utils.utils_noise import pair_selection_v1

print(
    inspect.getsource(
        pair_selection_v1
    )
)

def pair_selection_v1(k_val, test_loader, labels, class_num, cos_t, knn_times, train_features, balance_class=True,
                   sel_ratio=0, corrected=False):
    '''
    k_val:  neighbors number of knn
    labels: pseudo-labels obtained from feature prototypes
    '''
    similarity_graph_all = torch.zeros(len(test_loader.dataset), len(test_loader.dataset))
    train_noisy_labels = labels.clone().cuda()
    train_labels = labels.clone().cuda()
    discrepancy_measure = torch.zeros((len(test_loader.dataset),)).cuda()
    agreement_measure = torch.zeros((len(test_loader.dataset),))

    with torch.no_grad():
        for i in range(knn_times):
            print(f'starting the {i+1}st knn....')
            retrieval_one_hot_train = torch.zeros(k_val, class_num).cuda()
            train_new_labels = train_labels.clone()
            for batch_idx, (_, _, index) in enumerate(test_loader):
                batch_size = index.size(0)
                features = train_features[index]

     

In [23]:
# ============================================================
# CELL 18 — DECOMPOSE LOCKED SOURCE MODEL FOR UPA PORT
# NO TRAINING
# ============================================================

import tensorflow as tf
import numpy as np

source_upa_model = tf.keras.models.load_model(
    SOURCE_MODEL_PATH
)

print("=" * 72)
print("LOCKED SOURCE MODEL")
print("=" * 72)

source_upa_model.summary()

print("\nTop-level layers:")
for i, layer in enumerate(source_upa_model.layers):
    print(
        i,
        layer.name,
        type(layer).__name__,
        layer.output_shape
        if hasattr(layer, "output_shape")
        else ""
    )

# ------------------------------------------------------------
# Locate backbone / GAP / classifier
# ------------------------------------------------------------

resnet_layer = None
gap_layer = None
classifier_layer = None

for layer in source_upa_model.layers:

    if isinstance(layer, tf.keras.Model):
        if "resnet" in layer.name.lower():
            resnet_layer = layer

    if isinstance(
        layer,
        tf.keras.layers.GlobalAveragePooling2D
    ):
        gap_layer = layer

    if (
        isinstance(layer, tf.keras.layers.Dense)
        and layer.units == NUM_CLASSES
    ):
        classifier_layer = layer


assert resnet_layer is not None, \
    "ResNet50 backbone not found."

assert gap_layer is not None, \
    "GlobalAveragePooling layer not found."

assert classifier_layer is not None, \
    "3-class classifier not found."


print("\nDetected components:")
print("Backbone   :", resnet_layer.name)
print("GAP        :", gap_layer.name)
print("Classifier :", classifier_layer.name)

print("\nClassifier trainable:",
      classifier_layer.trainable)

print("Classifier kernel shape:",
      classifier_layer.kernel.shape)

print("Classifier bias shape:",
      classifier_layer.bias.shape)


# ============================================================
# FEATURE MODEL
# Output = 2048-D GAP representation
# before Dropout/Dense
# ============================================================

feature_model = tf.keras.Model(
    inputs=source_upa_model.input,
    outputs=gap_layer.output,
    name="UPA_shared_feature_extractor"
)

print("\nFeature dimensionality:",
      feature_model.output_shape)


# ============================================================
# VERIFY THAT WE CAN RECONSTRUCT ORIGINAL LOGITS
# FROM GAP FEATURES + ORIGINAL DENSE CLASSIFIER
# ============================================================

sample_ds = make_unlabeled_ds_exact(
    hard_test_df.iloc[:32].copy()
)

sample_images = next(iter(sample_ds))

features = feature_model(
    sample_images,
    training=False
)

# Dropout has no effect at inference.
manual_logits = classifier_layer(
    features
)

original_logits = source_upa_model(
    sample_images,
    training=False
)

max_difference = tf.reduce_max(
    tf.abs(
        manual_logits
        - original_logits
    )
).numpy()

print(
    "\nMaximum reconstructed-logit difference:",
    max_difference
)

assert max_difference < 1e-5, \
    "Feature/classifier decomposition does not reproduce source logits."

print("\n" + "=" * 72)
print("SOURCE ARCHITECTURE DECOMPOSITION VERIFIED")
print("=" * 72)

LOCKED SOURCE MODEL


Model: "ResNet50_logit_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 224, 224)  │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 224, 224)  │          0 │ input_layer[0][0] │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 224, 224)  │          0 │ input_layer[0][0] │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 224, 224,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 224, 224,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add[0][0]         │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 2048)      │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ logits (Dense)      │ (None, 3)         │      6,147 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 70,569,099 (269.20 MB)

 Trainable params: 23,487,619 (89.60 MB)

 Non-trainable params: 106,240 (415.00 KB)

 Optimizer params: 46,975,240 (179.20 MB)


Top-level layers:
0 input_layer InputLayer 
1 resnet50 Functional (None, 7, 7, 2048)
2 global_average_pooling2d GlobalAveragePooling2D 
3 dropout Dropout 
4 logits Dense 

Detected components:
Backbone   : resnet50
GAP        : global_average_pooling2d
Classifier : logits

Classifier trainable: True
Classifier kernel shape: (2048, 3)
Classifier bias shape: (3,)

Feature dimensionality: (None, 2048)

Maximum reconstructed-logit difference: 0.0

SOURCE ARCHITECTURE DECOMPOSITION VERIFIED


In [24]:
# ============================================================
# CELL 19 — UPA PROTOTYPE-REFINED PSEUDO-LABELS
# SHARED SOURCE ARCHITECTURE — NO TRAINING
# ============================================================

import numpy as np
import tensorflow as tf
from scipy.spatial.distance import cdist
from sklearn.metrics import confusion_matrix

UPA_K = 4
UPA_KNN_TIMES = 2
UPA_SEL_RATIO = 0.60
UPA_COS_T = 5.0
UPA_NUM_CLASSES = 3

# ------------------------------------------------------------
# Exact locked target-training set
# ------------------------------------------------------------
upa_target_df = (
    all_seed_splits[42]["target_hard_train_active"]
    .copy()
    .reset_index(drop=True)
)

assert len(upa_target_df) == 7200

# ------------------------------------------------------------
# Inference dataset — NO shuffle
# ------------------------------------------------------------
upa_infer_ds = make_unlabeled_ds_exact(
    upa_target_df
)

# ------------------------------------------------------------
# Extract GAP features and source logits
# ------------------------------------------------------------
print("=" * 78)
print("EXTRACTING LOCKED SOURCE FEATURES")
print("=" * 78)

all_features_raw = feature_model.predict(
    upa_infer_ds,
    verbose=1
).astype("float32")

all_logits = source_upa_model.predict(
    upa_infer_ds,
    verbose=1
).astype("float32")

all_probs = tf.nn.softmax(
    all_logits,
    axis=1
).numpy().astype("float32")

source_predictions = np.argmax(
    all_probs,
    axis=1
).astype(np.int64)

print("\nRaw GAP feature shape :", all_features_raw.shape)
print("Logit shape           :", all_logits.shape)
print("Probability shape     :", all_probs.shape)

assert all_features_raw.shape == (7200, 2048)
assert all_logits.shape == (7200, 3)

# ============================================================
# OFFICIAL UPA / SHOT-STYLE obtain_label()
#
# For cosine distance:
#   append constant 1
#   normalize entire augmented feature
#   build soft prototypes
#   nearest prototype assignment
#   one hard refinement round
# ============================================================

ones = np.ones(
    (all_features_raw.shape[0], 1),
    dtype=np.float32
)

features_aug = np.concatenate(
    [all_features_raw, ones],
    axis=1
)

norms = np.linalg.norm(
    features_aug,
    ord=2,
    axis=1,
    keepdims=True
)

features_aug = (
    features_aug /
    np.maximum(norms, 1e-12)
).astype("float32")

K = all_probs.shape[1]

# ------------------------------------------------------------
# Initial soft class prototypes
# ------------------------------------------------------------
initc = (
    all_probs.T @ features_aug
)

initc = (
    initc /
    (
        1e-8
        + all_probs.sum(axis=0)[:, None]
    )
)

# ------------------------------------------------------------
# Same label-set logic as official implementation
# ------------------------------------------------------------
cls_count = np.eye(
    K,
    dtype=np.float32
)[source_predictions].sum(axis=0)

labelset = np.where(
    cls_count > 0
)[0]

print("\nClasses represented by source predictions:")
print(labelset)

assert len(labelset) == UPA_NUM_CLASSES, (
    "At least one class disappeared from source predictions."
)

# ------------------------------------------------------------
# First nearest-prototype assignment
# ------------------------------------------------------------
dist = cdist(
    features_aug,
    initc[labelset],
    metric="cosine"
)

pseudo_labels = labelset[
    np.argmin(dist, axis=1)
].astype(np.int64)

# ------------------------------------------------------------
# Official code performs ONE hard refinement round
# ------------------------------------------------------------
aff = np.eye(
    K,
    dtype=np.float32
)[pseudo_labels]

initc = (
    aff.T @ features_aug
)

initc = (
    initc /
    (
        1e-8
        + aff.sum(axis=0)[:, None]
    )
)

dist = cdist(
    features_aug,
    initc[labelset],
    metric="cosine"
)

pseudo_labels = labelset[
    np.argmin(dist, axis=1)
].astype(np.int64)

# ------------------------------------------------------------
# pair_selection_v1 receives features with final appended
# coordinate REMOVED, exactly like UPA obtain_label()
# ------------------------------------------------------------
upa_knn_features = (
    features_aug[:, :-1]
    .astype("float32")
)

print("\n" + "=" * 78)
print("UPA PROTOTYPE REFINEMENT COMPLETE")
print("=" * 78)

print("\nOriginal source-prediction counts:")
print(
    pd.Series(source_predictions)
    .value_counts()
    .sort_index()
)

print("\nPrototype-refined pseudo-label counts:")
print(
    pd.Series(pseudo_labels)
    .value_counts()
    .sort_index()
)

changed = np.sum(
    source_predictions != pseudo_labels
)

print(
    "\nPredictions changed by prototype refinement:",
    changed,
    f"({100 * changed / len(pseudo_labels):.2f}%)"
)

print("\nKNN feature shape:", upa_knn_features.shape)

# Save fixed starting pseudo-label state
np.save(
    CONTROL_DIR / "upa_initial_features_seed42.npy",
    upa_knn_features
)

np.save(
    CONTROL_DIR / "upa_initial_pseudo_labels_seed42.npy",
    pseudo_labels
)

EXTRACTING LOCKED SOURCE FEATURES
225/225 ━━━━━━━━━━━━━━━━━━━━ 22s 83ms/step
225/225 ━━━━━━━━━━━━━━━━━━━━ 23s 87ms/step

Raw GAP feature shape : (7200, 2048)
Logit shape           : (7200, 3)
Probability shape     : (7200, 3)

Classes represented by source predictions:
[0 1 2]

UPA PROTOTYPE REFINEMENT COMPLETE

Original source-prediction counts:
0    3101
1    2498
2    1601
Name: count, dtype: int64

Prototype-refined pseudo-label counts:
0    2281
1    2451
2    2468
Name: count, dtype: int64

Predictions changed by prototype refinement: 1054 (14.64%)

KNN feature shape: (7200, 2048)


In [25]:
# ============================================================
# CELL 20 — UPA KNN UNCERTAINTY / CLEAN-SAMPLE SELECTION
#
# Algorithmically follows official pair_selection_v1().
# No model training.
# ============================================================

import numpy as np
import pandas as pd

def upa_pair_selection_shared(
    features,
    labels,
    class_num=3,
    k_val=4,
    cos_t=5.0,
    knn_times=2,
    sel_ratio=0.60,
    balance_class=True,
    batch_size=256
):
    """
    Shared-architecture implementation of UPA pair_selection_v1.

    Preserves the selection-relevant operations of the official code.

    features:
        N x D features returned by UPA-style obtain_label().
    labels:
        prototype-refined pseudo-labels.
    """

    features = np.asarray(
        features,
        dtype=np.float32
    )

    labels = np.asarray(
        labels,
        dtype=np.int64
    )

    N = len(labels)

    train_labels = labels.copy()

    discrepancy_measure = np.zeros(
        N,
        dtype=np.float32
    )

    agreement_measure = np.zeros(
        N,
        dtype=np.float32
    )

    all_indices = np.arange(N)

    # ========================================================
    # Iterative KNN label propagation
    # ========================================================
    for iteration in range(knn_times):

        print(
            f"\nStarting UPA KNN pass "
            f"{iteration + 1}/{knn_times}"
        )

        # Official:
        # train_new_labels = train_labels.clone()
        # before processing this KNN pass.
        train_new_labels = train_labels.copy()

        for start in range(
            0,
            N,
            batch_size
        ):

            end = min(
                start + batch_size,
                N
            )

            index = all_indices[start:end]

            batch_features = features[index]

            # ------------------------------------------------
            # Official:
            # dist = torch.mm(features,
            #                 train_features.t())
            # ------------------------------------------------
            similarity = (
                batch_features
                @ features.T
            )

            # Remove self-neighbor exactly as official code.
            similarity[
                np.arange(len(index)),
                index
            ] = -1.0

            # ------------------------------------------------
            # Top-k largest similarities
            # ------------------------------------------------
            # argpartition gives the same neighbor SET;
            # sort afterward for deterministic ordering.
            top_idx_unsorted = np.argpartition(
                -similarity,
                kth=k_val - 1,
                axis=1
            )[:, :k_val]

            top_sim_unsorted = np.take_along_axis(
                similarity,
                top_idx_unsorted,
                axis=1
            )

            order = np.argsort(
                -top_sim_unsorted,
                axis=1
            )

            neighbor_idx = np.take_along_axis(
                top_idx_unsorted,
                order,
                axis=1
            )

            neighbor_sim = np.take_along_axis(
                top_sim_unsorted,
                order,
                axis=1
            )

            # ------------------------------------------------
            # Retrieve labels from frozen labels for this pass
            # ------------------------------------------------
            neighbor_labels = (
                train_new_labels[
                    neighbor_idx
                ]
            )

            # ------------------------------------------------
            # exp(similarity / cos_t)
            # ------------------------------------------------
            neighbor_weight = np.exp(
                neighbor_sim / cos_t
            ).astype("float32")

            # ------------------------------------------------
            # Weighted class probabilities
            # ------------------------------------------------
            probs = np.zeros(
                (
                    len(index),
                    class_num
                ),
                dtype=np.float32
            )

            for c in range(class_num):

                probs[:, c] = np.sum(
                    neighbor_weight
                    * (
                        neighbor_labels == c
                    ),
                    axis=1
                )

            probs = (
                probs /
                np.maximum(
                    probs.sum(
                        axis=1,
                        keepdims=True
                    ),
                    1e-12
                )
            )

            # ------------------------------------------------
            # Discrepancy is calculated against ORIGINAL
            # prototype label, as in official UPA.
            # ------------------------------------------------
            prob_temp = probs[
                np.arange(len(index)),
                labels[index]
            ]

            prob_temp = np.clip(
                prob_temp,
                1e-4,
                1.0 - 1e-4
            )

            discrepancy_measure[index] = (
                -np.log(prob_temp)
            )

            # ------------------------------------------------
            # Updated KNN label
            # ------------------------------------------------
            new_labels = np.argmax(
                probs,
                axis=1
            )

            train_labels[index] = (
                new_labels
            )

            agreement_measure[index] = (
                new_labels
                == labels[index]
            ).astype(np.float32)

    # ========================================================
    # UPA class-balanced clean selection
    # ========================================================
    if balance_class:

        selected_examples = np.zeros(
            N,
            dtype=np.float32
        )

        for c in range(class_num):

            idx_class = np.where(
                labels == c
            )[0]

            num_per_class = len(
                idx_class
            )

            k_corrected = int(
                sel_ratio
                * num_per_class
            )

            if k_corrected >= 1:

                discrepancy_class = (
                    discrepancy_measure[
                        idx_class
                    ]
                )

                relative_order = np.argsort(
                    discrepancy_class
                )

                chosen = idx_class[
                    relative_order[
                        :k_corrected
                    ]
                ]

                selected_examples[
                    chosen
                ] = 1.0

    else:

        selected_examples = (
            agreement_measure.copy()
        )

    selected_indices = np.where(
        selected_examples > 0
    )[0]

    return {
        "selected_examples": selected_examples,
        "selected_indices": selected_indices,
        "discrepancy": discrepancy_measure,
        "knn_updated_labels": train_labels,
        "original_pseudo_labels": labels.copy()
    }


# ============================================================
# RUN OFFICIAL OFFICE-HOME HYPERPARAMETERS
# ============================================================

upa_selection = upa_pair_selection_shared(
    features=upa_knn_features,
    labels=pseudo_labels,
    class_num=UPA_NUM_CLASSES,
    k_val=UPA_K,
    cos_t=UPA_COS_T,
    knn_times=UPA_KNN_TIMES,
    sel_ratio=UPA_SEL_RATIO,
    balance_class=True,
    batch_size=256
)

selected_mask = (
    upa_selection[
        "selected_examples"
    ] > 0
)

selected_indices = (
    upa_selection[
        "selected_indices"
    ]
)

print("\n" + "=" * 78)
print("UPA CLEAN-SAMPLE SELECTION")
print("=" * 78)

print(
    "\nTotal target samples:",
    len(pseudo_labels)
)

print(
    "Selected clean samples:",
    len(selected_indices)
)

print(
    "Selected fraction:",
    f"{100 * len(selected_indices) / len(pseudo_labels):.2f}%"
)

# ------------------------------------------------------------
# Per-class selection
# ------------------------------------------------------------
selection_table = []

for c, class_name in enumerate(
    CLASS_NAMES
):

    class_mask = (
        pseudo_labels == c
    )

    total_c = int(
        class_mask.sum()
    )

    selected_c = int(
        np.sum(
            class_mask
            & selected_mask
        )
    )

    selection_table.append({
        "class_id": c,
        "class_name": class_name,
        "pseudo_total": total_c,
        "selected": selected_c,
        "selected_fraction": (
            selected_c / total_c
            if total_c > 0
            else np.nan
        )
    })

selection_df = pd.DataFrame(
    selection_table
)

print("\nPer-class UPA clean selection:")
display(selection_df)

# ------------------------------------------------------------
# Save diagnostics
# ------------------------------------------------------------
upa_diag_df = upa_target_df.copy()

upa_diag_df[
    "source_prediction"
] = source_predictions

upa_diag_df[
    "upa_pseudo_label"
] = pseudo_labels

upa_diag_df[
    "upa_knn_updated_label"
] = upa_selection[
    "knn_updated_labels"
]

upa_diag_df[
    "upa_discrepancy"
] = upa_selection[
    "discrepancy"
]

upa_diag_df[
    "upa_selected"
] = selected_mask.astype(int)

upa_diag_df.to_csv(
    CONTROL_DIR /
    "upa_initial_selection_seed42.csv",
    index=False
)

print(
    "\nSaved:",
    CONTROL_DIR /
    "upa_initial_selection_seed42.csv"
)


Starting UPA KNN pass 1/2

Starting UPA KNN pass 2/2

UPA CLEAN-SAMPLE SELECTION

Total target samples: 7200
Selected clean samples: 4318
Selected fraction: 59.97%

Per-class UPA clean selection:


,class_id,class_name,pseudo_total,selected,selected_fraction
0,0,Early Blight,2281,1368,0.599737
1,1,Late Blight,2451,1470,0.599755
2,2,Healthy,2468,1480,0.599676



Saved: /kaggle/working/external_sfuda_baseline/controlled_seed42/upa_initial_selection_seed42.csv


In [28]:
# ============================================================
# CELL 21 — SHARED-ARCHITECTURE UPA TRAINING UTILITIES
# ============================================================

import math
import numpy as np
import pandas as pd
import tensorflow as tf

UPA_EPOCHS = 15
UPA_BATCH = 32          # keep shared experiment batch size
UPA_BASE_LR = 1e-3      # official 1e-2 × lr_decay1=0.1
UPA_WEIGHT_DECAY = 1e-3

UPA_PAR_CLS = 0.30
UPA_PAR_ENT = 1.00
UPA_PAR_SUCL = 1.00
UPA_SUCL_T = 0.10

# ------------------------------------------------------------
# Frozen source classifier
# ------------------------------------------------------------
upa_model = tf.keras.models.load_model(
    SOURCE_MODEL_PATH
)

upa_resnet = None
upa_gap = None
upa_classifier = None

for layer in upa_model.layers:

    if isinstance(layer, tf.keras.Model) and \
       "resnet" in layer.name.lower():
        upa_resnet = layer

    elif isinstance(
        layer,
        tf.keras.layers.GlobalAveragePooling2D
    ):
        upa_gap = layer

    elif isinstance(
        layer,
        tf.keras.layers.Dense
    ) and layer.units == NUM_CLASSES:
        upa_classifier = layer


assert upa_resnet is not None
assert upa_gap is not None
assert upa_classifier is not None

upa_classifier.trainable = False

# ------------------------------------------------------------
# Feature/logit forward path
# Dropout omitted during adaptation, matching deterministic
# source-classifier use in UPA.
# ------------------------------------------------------------
def upa_forward(images, training=True):

    x = tf.keras.applications.resnet50.preprocess_input(
        images
    )

    feat_map = upa_resnet(
        x,
        training=training
    )

    features = upa_gap(
        feat_map
    )

    logits = upa_classifier(
        features
    )

    return features, logits


# ============================================================
# AUGMENTATIONS
# ============================================================

def decode_raw_image(path):

    img = tf.io.read_file(path)

    img = tf.image.decode_image(
        img,
        channels=3,
        expand_animations=False
    )

    img = tf.cast(
        img,
        tf.float32
    )

    return img


def weak_aug(img):

    img = tf.image.resize(
        img,
        [256, 256]
    )

    img = tf.image.random_crop(
        img,
        [224, 224, 3]
    )

    img = tf.image.random_flip_left_right(
        img
    )

    return img


def random_resized_crop(img):

    img = tf.image.resize(
        img,
        [256, 256]
    )

    scale = tf.random.uniform(
        [],
        0.20,
        1.00
    )

    crop_size = tf.cast(
        tf.sqrt(scale) * 256.0,
        tf.int32
    )

    crop_size = tf.clip_by_value(
        crop_size,
        64,
        256
    )

    cropped = tf.image.random_crop(
        img,
        [crop_size, crop_size, 3]
    )

    cropped = tf.image.resize(
        cropped,
        [224, 224]
    )

    return cropped


def strong_aug(img):

    img = random_resized_crop(
        img
    )

    # Color jitter approximation
    if tf.random.uniform([]) < 0.8:

        img = tf.image.random_brightness(
            img,
            max_delta=0.4 * 255.0
        )

        img = tf.image.random_contrast(
            img,
            lower=0.6,
            upper=1.4
        )

        img = tf.image.random_saturation(
            img,
            lower=0.6,
            upper=1.4
        )

        img = tf.image.random_hue(
            img / 255.0,
            max_delta=0.1
        ) * 255.0

    if tf.random.uniform([]) < 0.2:

        gray = tf.image.rgb_to_grayscale(
            img
        )

        img = tf.image.grayscale_to_rgb(
            gray
        )

    img = tf.image.random_flip_left_right(
        img
    )

    img = tf.clip_by_value(
        img,
        0.0,
        255.0
    )

    return img


# ============================================================
# TRAIN DATASET
# ============================================================

def load_two_views(path, index):

    img = decode_raw_image(
        path
    )

    weak = weak_aug(
        img
    )

    strong = strong_aug(
        img
    )

    return (
        weak,
        strong,
        index
    )


def make_upa_two_view_ds(df):

    indices = np.arange(
        len(df),
        dtype=np.int32
    )

    ds = tf.data.Dataset.from_tensor_slices(
        (
            df["file_path"].values,
            indices
        )
    )

    ds = ds.shuffle(
        len(df),
        seed=42,
        reshuffle_each_iteration=True
    )

    ds = ds.map(
        load_two_views,
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(
        UPA_BATCH,
        drop_remainder=False
    )

    ds = ds.prefetch(
        AUTOTUNE
    )

    return ds


upa_train_ds = make_upa_two_view_ds(
    upa_target_df
)


# ============================================================
# INFORMATION MAXIMIZATION LOSS
# ============================================================

def entropy_from_probs(p):

    return -tf.reduce_sum(
        p * tf.math.log(
            p + 1e-5
        ),
        axis=1
    )


def information_max_loss(logits):

    probs = tf.nn.softmax(
        logits,
        axis=1
    )

    entropy_loss = tf.reduce_mean(
        entropy_from_probs(
            probs
        )
    )

    mean_prob = tf.reduce_mean(
        probs,
        axis=0
    )

    global_entropy = -tf.reduce_sum(
        mean_prob
        * tf.math.log(
            mean_prob + 1e-5
        )
    )

    return (
        entropy_loss
        - global_entropy
    )


# ============================================================
# UPA CONTRASTIVE LOSS
# ============================================================

def upa_contrastive_loss(
    weak_logits,
    strong_logits,
    batch_indices,
    pseudo_labels_tf,
    selected_mask_tf
):

    q = tf.math.l2_normalize(
        weak_logits,
        axis=1
    )

    k = tf.math.l2_normalize(
        strong_logits,
        axis=1
    )

    embeds = tf.concat(
        [q, k],
        axis=0
    )

    pairwise = tf.matmul(
        embeds,
        embeds,
        transpose_b=True
    )

    logits = (
        pairwise
        / UPA_SUCL_T
    )

    bsz = tf.shape(
        weak_logits
    )[0]

    batch_labels = tf.gather(
        pseudo_labels_tf,
        batch_indices
    )

    batch_selected = tf.gather(
        selected_mask_tf,
        batch_indices
    )

    same_label = tf.equal(
        batch_labels[:, None],
        batch_labels[None, :]
    )

    both_selected = tf.logical_and(
        batch_selected[:, None],
        batch_selected[None, :]
    )

    supervised_mask = tf.cast(
        tf.logical_and(
            same_label,
            both_selected
        ),
        tf.float32
    )

    supervised_mask = tf.linalg.set_diag(
        supervised_mask,
        tf.zeros(
            [bsz],
            tf.float32
        )
    )

    # Repeat exactly across weak/strong blocks.
    mask_sup = tf.tile(
        supervised_mask,
        [2, 2]
    )

    identity_b = tf.eye(
        bsz,
        dtype=tf.float32
    )

    mask_unsup = tf.tile(
        identity_b,
        [2, 2]
    )

    eye_2b = tf.eye(
        2 * bsz,
        dtype=tf.float32
    )

    logits_mask = (
        1.0 - eye_2b
    )

    mask_sup = (
        mask_sup
        * logits_mask
    )

    mask_unsup = (
        mask_unsup
        * logits_mask
    )

    exp_logits = (
        tf.exp(logits)
        * logits_mask
    )

    log_prob = (
        logits
        - tf.math.log(
            tf.reduce_sum(
                exp_logits,
                axis=1,
                keepdims=True
            )
            + 1e-7
        )
    )

    denom = (
        tf.reduce_sum(
            mask_sup,
            axis=1
        )
        +
        tf.reduce_sum(
            mask_unsup,
            axis=1
        )
        + 1e-7
    )

    mean_sup = (
        tf.reduce_sum(
            mask_sup
            * log_prob,
            axis=1
        )
        / denom
    )

    mean_unsup = (
        tf.reduce_sum(
            mask_unsup
            * log_prob,
            axis=1
        )
        / denom
    )

    loss_all = (
        -mean_sup
        -mean_unsup
    )

    first_half = loss_all[:bsz]
    second_half = loss_all[bsz:]

    per_sample = (
        first_half
        + second_half
    ) / 2.0

    has_positive = (
        tf.reduce_sum(
            mask_sup[:bsz],
            axis=1
        ) > 0
    )

    per_sample = (
        tf.cast(
            has_positive,
            tf.float32
        )
        * per_sample
    )

    return tf.reduce_mean(
        per_sample
    )


print("=" * 72)
print("UPA TRAINING UTILITIES READY")
print("=" * 72)
print("Epochs       :", UPA_EPOCHS)
print("Batch size   :", UPA_BATCH)
print("Encoder LR   :", UPA_BASE_LR)
print("Classifier frozen:", not upa_classifier.trainable)

UPA TRAINING UTILITIES READY
Epochs       : 15
Batch size   : 32
Encoder LR   : 0.001
Classifier frozen: True


In [31]:
# ============================================================
# PRE-CELL 22 — CLEANUP + SAFE MODEL LOAD TEST
# ============================================================

import gc
import tensorflow as tf

for name in [
    "upa_model",
    "source_upa_model",
    "feature_model"
]:
    if name in globals():
        try:
            del globals()[name]
        except:
            pass

gc.collect()
tf.keras.backend.clear_session()

upa_model = tf.keras.models.load_model(
    SOURCE_MODEL_PATH,
    compile=False
)

print("SUCCESS")
print("Model loaded with compile=False.")
print("Parameters:", upa_model.count_params())

SUCCESS
Model loaded with compile=False.
Parameters: 23593859


In [32]:
upa_model = tf.keras.models.load_model(
    SOURCE_MODEL_PATH,
    compile=False
)

In [3]:
from pathlib import Path

CONTROL_DIR = Path(
    "/kaggle/working/external_sfuda_baseline/controlled_seed42"
)

SOURCE_MODEL_PATH = (
    CONTROL_DIR /
    "ResNet50_reproduced_source_seed42.keras"
)

print("SOURCE_MODEL_PATH =", SOURCE_MODEL_PATH)
print("Exists:", SOURCE_MODEL_PATH.exists())

SOURCE_MODEL_PATH = /kaggle/working/external_sfuda_baseline/controlled_seed42/ResNet50_reproduced_source_seed42.keras
Exists: True


In [8]:
# ============================================================
# RECOVER EXACT SEED-42 HARD-TARGET TRAIN SPLIT
# ============================================================

from pathlib import Path
import pandas as pd

SEED42_TRAIN_CSV = Path(
    "/kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/"
    "TWD_paper_final_package_FIXED_20260905_234741/"
    "raw_result_files/target_hard_train_unlabeled.csv"
)

assert SEED42_TRAIN_CSV.exists(), (
    f"File not found:\n{SEED42_TRAIN_CSV}"
)

upa_target_df = pd.read_csv(SEED42_TRAIN_CSV)

print("Loaded:", SEED42_TRAIN_CSV)
print("Rows:", len(upa_target_df))
print("Columns:", list(upa_target_df.columns))

assert len(upa_target_df) == 7200, (
    f"Expected 7200 Seed-42 target samples, got {len(upa_target_df)}"
)

assert "file_path" in upa_target_df.columns, (
    "Expected a 'file_path' column."
)

print("\nSUCCESS")
print("upa_target_df =", len(upa_target_df), "samples")
print(upa_target_df.head(3))

Loaded: /kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/TWD_paper_final_package_FIXED_20260905_234741/raw_result_files/target_hard_train_unlabeled.csv
Rows: 7200
Columns: ['file_path', 'dataset_role', 'raw_class', 'canonical_class', 'label']

SUCCESS
upa_target_df = 7200 samples
                                           file_path       dataset_role  \
0  /kaggle/input/datasets/manhhoangvan/irishpotat...  target_hard_irish   
1  /kaggle/input/datasets/manhhoangvan/irishpotat...  target_hard_irish   
2  /kaggle/input/datasets/manhhoangvan/irishpotat...  target_hard_irish   

  raw_class canonical_class  label  
0   healthy         Healthy      2  
1   healthy         Healthy      2  
2   healthy         Healthy      2  


In [9]:
# ============================================================
# CELL 22 — FINAL MEMORY-SAFE CONTROLLED UPA ADAPTATION
# ============================================================

import gc
import math
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from scipy.spatial.distance import cdist


# ============================================================
# 0. FIXED PATHS / CONSTANTS
# ============================================================

CONTROL_DIR = Path(
    "/kaggle/working/external_sfuda_baseline/controlled_seed42"
)

SOURCE_MODEL_PATH = (
    CONTROL_DIR /
    "ResNet50_reproduced_source_seed42.keras"
)

NUM_CLASSES = 3
IMG_SIZE = 224

# Training batch remains locked
UPA_BATCH = 32

# Small inference batch only for prototype refresh
UPA_INFER_BATCH = 8

UPA_EPOCHS = 15

UPA_K = 4
UPA_KNN_TIMES = 2
UPA_SEL_RATIO = 0.60
UPA_COS_T = 5.0

UPA_BASE_LR = 1e-3
UPA_WEIGHT_DECAY = 1e-3

UPA_PAR_ENT = 1.0
UPA_PAR_SUCL = 1.0
UPA_PAR_CLS = 0.30
UPA_SUCL_TEMP = 0.10


assert SOURCE_MODEL_PATH.exists(), (
    f"Source model not found:\n{SOURCE_MODEL_PATH}"
)

assert "upa_target_df" in globals(), (
    "upa_target_df is not defined. "
    "Rerun the exact saved Seed-42 split-loading cell first."
)

assert len(upa_target_df) == 7200, (
    f"Expected 7200 target training samples, "
    f"found {len(upa_target_df)}"
)

print("Source checkpoint:", SOURCE_MODEL_PATH)
print("Target samples:", len(upa_target_df))


# ============================================================
# 1. MEMORY CLEANUP
# ============================================================

gc.collect()
tf.keras.backend.clear_session()


# ============================================================
# 2. LOAD SOURCE MODEL WITHOUT OLD OPTIMIZER
# ============================================================

upa_model = tf.keras.models.load_model(
    SOURCE_MODEL_PATH,
    compile=False
)

upa_resnet = None
upa_gap = None
upa_classifier = None

for layer in upa_model.layers:

    if (
        isinstance(layer, tf.keras.Model)
        and "resnet" in layer.name.lower()
    ):
        upa_resnet = layer

    elif isinstance(
        layer,
        tf.keras.layers.GlobalAveragePooling2D
    ):
        upa_gap = layer

    elif (
        isinstance(layer, tf.keras.layers.Dense)
        and layer.units == NUM_CLASSES
    ):
        upa_classifier = layer


assert upa_resnet is not None
assert upa_gap is not None
assert upa_classifier is not None

# Original classifier stays frozen
upa_classifier.trainable = False


def upa_forward(images, training=False):

    x = tf.keras.applications.resnet50.preprocess_input(
        images
    )

    feat_map = upa_resnet(
        x,
        training=training
    )

    features = upa_gap(
        feat_map
    )

    logits = upa_classifier(
        features
    )

    return features, logits


print("\n" + "=" * 78)
print("CONTROLLED UPA MODEL LOADED")
print("=" * 78)

print("Backbone   :", upa_resnet.name)
print("Classifier :", upa_classifier.name)
print("Classifier frozen:", not upa_classifier.trainable)


# ============================================================
# 3. RAW IMAGE LOADER
# ============================================================

def decode_raw_image(file_path):

    img = tf.io.read_file(
        file_path
    )

    img = tf.image.decode_image(
        img,
        channels=3,
        expand_animations=False
    )

    img.set_shape(
        [None, None, 3]
    )

    img = tf.image.resize(
        img,
        [IMG_SIZE, IMG_SIZE]
    )

    return tf.cast(
        img,
        tf.float32
    )


# ============================================================
# 4. MEMORY-SAFE INFERENCE DATASET
# ============================================================

def make_upa_inference_ds(df):

    ds = tf.data.Dataset.from_tensor_slices(
        df["file_path"].astype(str).values
    )

    ds = ds.map(
        decode_raw_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(
        UPA_INFER_BATCH,
        drop_remainder=False
    )

    # deliberately small prefetch
    ds = ds.prefetch(1)

    return ds


# ============================================================
# 5. WEAK AUGMENTATION
# ============================================================

def weak_aug(img):

    img = tf.image.resize(
        img,
        [256, 256]
    )

    img = tf.image.random_crop(
        img,
        [224, 224, 3]
    )

    img = tf.image.random_flip_left_right(
        img
    )

    return tf.cast(
        img,
        tf.float32
    )


# ============================================================
# 6. STRONG AUGMENTATION
# ============================================================

def strong_aug(img):

    img = tf.image.resize(
        img,
        [256, 256]
    )

    # Random crop approximation to UPA/MoCo-v2
    crop_size = tf.random.uniform(
        [],
        minval=180,
        maxval=257,
        dtype=tf.int32
    )

    img = tf.image.random_crop(
        img,
        [crop_size, crop_size, 3]
    )

    img = tf.image.resize(
        img,
        [224, 224]
    )

    img = tf.image.random_flip_left_right(
        img
    )

    # Color jitter
    img = tf.image.random_brightness(
        img,
        max_delta=0.4 * 255.0
    )

    img = tf.image.random_contrast(
        img,
        lower=0.6,
        upper=1.4
    )

    img = tf.image.random_saturation(
        img,
        lower=0.6,
        upper=1.4
    )

    img = tf.clip_by_value(
        img,
        0.0,
        255.0
    )

    return tf.cast(
        img,
        tf.float32
    )


# ============================================================
# 7. TWO-VIEW TRAINING DATASET
# ============================================================

def make_upa_two_view_ds(df):

    paths = df["file_path"].astype(str).values
    indices = np.arange(len(df))

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, indices)
    )

    ds = ds.shuffle(
        len(df),
        seed=42,
        reshuffle_each_iteration=True
    )

    def load_two(path, idx):

        img = decode_raw_image(
            path
        )

        weak = weak_aug(
            img
        )

        strong = strong_aug(
            img
        )

        return (
            weak,
            strong,
            idx
        )

    ds = ds.map(
        load_two,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(
        UPA_BATCH,
        drop_remainder=False
    )

    ds = ds.prefetch(1)

    return ds


upa_train_ds = make_upa_two_view_ds(
    upa_target_df
)


# ============================================================
# 8. INFORMATION MAXIMIZATION LOSS
# ============================================================

def information_max_loss(logits):

    probs = tf.nn.softmax(
        logits,
        axis=1
    )

    entropy_each = -tf.reduce_sum(
        probs * tf.math.log(
            probs + 1e-5
        ),
        axis=1
    )

    entropy_mean = tf.reduce_mean(
        entropy_each
    )

    mean_prob = tf.reduce_mean(
        probs,
        axis=0
    )

    global_entropy = -tf.reduce_sum(
        mean_prob * tf.math.log(
            mean_prob + 1e-5
        )
    )

    return (
        entropy_mean
        - global_entropy
    )


# ============================================================
# 9. PROTOTYPE-REFINED PSEUDO-LABELS
# ============================================================

def obtain_upa_labels_current():

    infer_ds = make_upa_inference_ds(
        upa_target_df
    )

    all_features = []
    all_outputs = []

    for batch in infer_ds:

        features, logits = upa_forward(
            batch,
            training=False
        )

        all_features.append(
            features.numpy()
        )

        all_outputs.append(
            logits.numpy()
        )

    all_features = np.concatenate(
        all_features,
        axis=0
    ).astype("float32")

    all_outputs = np.concatenate(
        all_outputs,
        axis=0
    ).astype("float32")

    all_probs = tf.nn.softmax(
        all_outputs,
        axis=1
    ).numpy()

    initial_pred = np.argmax(
        all_probs,
        axis=1
    )

    # UPA / SHOT cosine feature preparation
    features_aug = np.concatenate(
        [
            all_features,
            np.ones(
                (len(all_features), 1),
                dtype=np.float32
            )
        ],
        axis=1
    )

    features_aug /= np.maximum(
        np.linalg.norm(
            features_aug,
            axis=1,
            keepdims=True
        ),
        1e-12
    )

    K = NUM_CLASSES

    # soft class prototypes
    initc = (
        all_probs.T
        @ features_aug
    )

    initc /= (
        1e-8
        + all_probs.sum(
            axis=0
        )[:, None]
    )

    cls_count = np.eye(K)[
        initial_pred
    ].sum(axis=0)

    labelset = np.where(
        cls_count > 0
    )[0]

    dist = cdist(
        features_aug,
        initc[labelset],
        metric="cosine"
    )

    pseudo = labelset[
        dist.argmin(axis=1)
    ]

    # one hard prototype refinement
    aff = np.eye(K)[
        pseudo
    ]

    initc = (
        aff.T
        @ features_aug
    )

    initc /= (
        1e-8
        + aff.sum(
            axis=0
        )[:, None]
    )

    dist = cdist(
        features_aug,
        initc[labelset],
        metric="cosine"
    )

    pseudo = labelset[
        dist.argmin(axis=1)
    ].astype(np.int64)

    # remove appended coordinate
    knn_features = (
        features_aug[:, :-1]
        .astype("float32")
    )

    return (
        pseudo,
        knn_features
    )


# ============================================================
# 10. MEMORY-SAFE UPA KNN SELECTION
# ============================================================

def upa_pair_selection_shared(
    features,
    labels,
    class_num,
    k_val=4,
    cos_t=5.0,
    knn_times=2,
    sel_ratio=0.60,
    balance_class=True,
    batch_size=256
):

    features = np.asarray(
        features,
        dtype=np.float32
    )

    original_labels = np.asarray(
        labels,
        dtype=np.int64
    )

    current_labels = original_labels.copy()

    n = len(features)

    final_discrepancy = None

    for pass_id in range(
        knn_times
    ):

        print(
            f"Starting UPA KNN pass "
            f"{pass_id + 1}/{knn_times}"
        )

        updated_labels = np.zeros(
            n,
            dtype=np.int64
        )

        discrepancy_all = np.zeros(
            n,
            dtype=np.float32
        )

        for start in range(
            0,
            n,
            batch_size
        ):

            end = min(
                start + batch_size,
                n
            )

            sims = (
                features[start:end]
                @ features.T
            )

            rows = np.arange(
                end - start
            )

            global_indices = np.arange(
                start,
                end
            )

            sims[
                rows,
                global_indices
            ] = -1.0

            knn_idx = np.argpartition(
                -sims,
                kth=k_val - 1,
                axis=1
            )[:, :k_val]

            knn_sim = np.take_along_axis(
                sims,
                knn_idx,
                axis=1
            )

            order = np.argsort(
                -knn_sim,
                axis=1
            )

            knn_idx = np.take_along_axis(
                knn_idx,
                order,
                axis=1
            )

            knn_sim = np.take_along_axis(
                knn_sim,
                order,
                axis=1
            )

            neigh_labels = current_labels[
                knn_idx
            ]

            weights = np.exp(
                knn_sim / cos_t
            )

            class_scores = np.zeros(
                (
                    end - start,
                    class_num
                ),
                dtype=np.float32
            )

            for c in range(
                class_num
            ):

                class_scores[:, c] = (
                    weights
                    * (
                        neigh_labels == c
                    )
                ).sum(axis=1)

            class_probs = (
                class_scores
                /
                np.maximum(
                    class_scores.sum(
                        axis=1,
                        keepdims=True
                    ),
                    1e-12
                )
            )

            original_batch_labels = (
                original_labels[
                    start:end
                ]
            )

            p_original = class_probs[
                rows,
                original_batch_labels
            ]

            p_original = np.clip(
                p_original,
                1e-4,
                1.0 - 1e-4
            )

            discrepancy_all[
                start:end
            ] = -np.log(
                p_original
            )

            updated_labels[
                start:end
            ] = np.argmax(
                class_probs,
                axis=1
            )

        current_labels = (
            updated_labels
        )

        final_discrepancy = (
            discrepancy_all
        )

    selected_examples = np.zeros(
        n,
        dtype=np.float32
    )

    if balance_class:

        for c in range(
            class_num
        ):

            class_indices = np.where(
                original_labels == c
            )[0]

            if len(class_indices) == 0:
                continue

            n_select = int(
                sel_ratio
                * len(class_indices)
            )

            class_order = np.argsort(
                final_discrepancy[
                    class_indices
                ]
            )

            chosen = class_indices[
                class_order[:n_select]
            ]

            selected_examples[
                chosen
            ] = 1.0

    else:

        n_select = int(
            sel_ratio * n
        )

        order = np.argsort(
            final_discrepancy
        )

        selected_examples[
            order[:n_select]
        ] = 1.0

    return {
        "selected_examples":
            selected_examples,

        "discrepancy":
            final_discrepancy,

        "knn_labels":
            current_labels
    }


# ============================================================
# 11. CONTRASTIVE LOSS
# ============================================================

def upa_contrastive_loss(
    weak_logits,
    strong_logits,
    batch_indices,
    pseudo_labels,
    selected_mask
):

    # Normalize classifier outputs
    z1 = tf.math.l2_normalize(
        weak_logits,
        axis=1
    )

    z2 = tf.math.l2_normalize(
        strong_logits,
        axis=1
    )

    batch_indices = tf.cast(
        batch_indices,
        tf.int32
    )

    batch_labels = tf.gather(
        pseudo_labels,
        batch_indices
    )

    batch_selected = tf.gather(
        selected_mask,
        batch_indices
    )

    # Similarity between weak/strong views
    sim = tf.matmul(
        z1,
        z2,
        transpose_b=True
    )

    sim = (
        sim /
        UPA_SUCL_TEMP
    )

    # numerical stabilization
    sim = (
        sim
        - tf.reduce_max(
            sim,
            axis=1,
            keepdims=True
        )
    )

    exp_sim = tf.exp(
        sim
    )

    labels_i = tf.expand_dims(
        batch_labels,
        1
    )

    labels_j = tf.expand_dims(
        batch_labels,
        0
    )

    positive_mask = tf.cast(
        tf.equal(
            labels_i,
            labels_j
        ),
        tf.float32
    )

    selected_pair_mask = tf.cast(
        tf.logical_and(
            tf.expand_dims(
                batch_selected,
                1
            ),
            tf.expand_dims(
                batch_selected,
                0
            )
        ),
        tf.float32
    )

    positive_mask *= (
        selected_pair_mask
    )

    # always preserve corresponding augmented view
    eye = tf.eye(
        tf.shape(sim)[0]
    )

    positive_mask = tf.maximum(
        positive_mask,
        eye
    )

    numerator = tf.reduce_sum(
        exp_sim
        * positive_mask,
        axis=1
    )

    denominator = tf.reduce_sum(
        exp_sim,
        axis=1
    )

    loss = -tf.math.log(
        (
            numerator + 1e-8
        )
        /
        (
            denominator + 1e-8
        )
    )

    return tf.reduce_mean(
        loss
    )


# ============================================================
# 12. GRADIENT HELPER
# ============================================================

def add_gradient_sets(
    grads_a,
    grads_b
):

    combined = []

    for ga, gb in zip(
        grads_a,
        grads_b
    ):

        if (
            ga is None
            and gb is None
        ):
            combined.append(
                None
            )

        elif ga is None:
            combined.append(
                gb
            )

        elif gb is None:
            combined.append(
                ga
            )

        else:
            combined.append(
                ga + gb
            )

    return combined


# ============================================================
# 13. QUICK MEMORY TEST BEFORE TRAINING
# ============================================================

print("\nRunning memory-safe inference test...")

test_ds = make_upa_inference_ds(
    upa_target_df
)

for batch in test_ds.take(1):

    features_test, logits_test = upa_forward(
        batch,
        training=False
    )

    print(
        "Input:",
        batch.shape
    )

    print(
        "Features:",
        features_test.shape
    )

    print(
        "Logits:",
        logits_test.shape
    )

del features_test
del logits_test

gc.collect()

print(
    "UPA inference memory test PASSED."
)


# ============================================================
# 14. OPTIMIZER
# ============================================================

optimizer_upa = tf.keras.optimizers.SGD(
    learning_rate=UPA_BASE_LR,
    momentum=0.9,
    nesterov=True
)

steps_per_epoch = int(
    np.ceil(
        len(upa_target_df)
        / UPA_BATCH
    )
)

max_steps = (
    UPA_EPOCHS
    * steps_per_epoch
)

global_step = 0
upa_history = []


# ============================================================
# 15. TRAINING LOOP
# ============================================================

for epoch in range(
    1,
    UPA_EPOCHS + 1
):

    print(
        "\n"
        + "=" * 78
    )

    print(
        f"UPA EPOCH "
        f"{epoch}/{UPA_EPOCHS}"
    )

    print(
        "=" * 78
    )

    # --------------------------------------------------------
    # Update prototype pseudo-labels
    # --------------------------------------------------------
    (
        epoch_pseudo,
        epoch_features
    ) = obtain_upa_labels_current()

    unique, counts = np.unique(
        epoch_pseudo,
        return_counts=True
    )

    print(
        "\nPseudo-label counts:"
    )

    for c, n in zip(
        unique,
        counts
    ):

        print(
            f"  class {c}: {n}"
        )

    # --------------------------------------------------------
    # UPA uncertainty selection
    # --------------------------------------------------------
    epoch_selection = (
        upa_pair_selection_shared(
            features=epoch_features,
            labels=epoch_pseudo,
            class_num=NUM_CLASSES,
            k_val=UPA_K,
            cos_t=UPA_COS_T,
            knn_times=UPA_KNN_TIMES,
            sel_ratio=UPA_SEL_RATIO,
            balance_class=True,
            batch_size=256
        )
    )

    selected_mask_np = (
        epoch_selection[
            "selected_examples"
        ] > 0
    )

    selected_indices_np = np.where(
        selected_mask_np
    )[0]

    print(
        "\nSelected:",
        len(selected_indices_np),
        "/",
        len(epoch_pseudo),
        f"({100 * len(selected_indices_np) / len(epoch_pseudo):.2f}%)"
    )

    assert (
        len(selected_indices_np)
        > 0
    )

    pseudo_tf = tf.constant(
        epoch_pseudo,
        dtype=tf.int32
    )

    selected_mask_tf = tf.constant(
        selected_mask_np,
        dtype=tf.bool
    )

    epoch_total = []
    epoch_im = []
    epoch_cls = []
    epoch_sucl = []

    # --------------------------------------------------------
    # MINI-BATCH TRAINING
    # --------------------------------------------------------
    for (
        weak_imgs,
        strong_imgs,
        batch_indices
    ) in upa_train_ds:

        global_step += 1

        lr_now = (
            UPA_BASE_LR
            * 0.5
            * (
                1.0
                + math.cos(
                    math.pi
                    * global_step
                    / max_steps
                )
            )
        )

        optimizer_upa.learning_rate.assign(
            lr_now
        )

        bsz = int(
            weak_imgs.shape[0]
        )

        train_vars = (
            upa_resnet.trainable_variables
        )

        # ====================================================
        # PART A:
        # INFORMATION MAX + CONTRASTIVE
        # ====================================================
        with tf.GradientTape() as tape_a:

            _, weak_logits = upa_forward(
                weak_imgs,
                training=True
            )

            _, strong_logits = upa_forward(
                strong_imgs,
                training=True
            )

            loss_im = (
                information_max_loss(
                    weak_logits
                )
            )

            loss_sucl = (
                upa_contrastive_loss(
                    weak_logits,
                    strong_logits,
                    batch_indices,
                    pseudo_tf,
                    selected_mask_tf
                )
            )

            loss_a = (
                UPA_PAR_ENT
                * loss_im
                +
                UPA_PAR_SUCL
                * loss_sucl
            )

            # L2 regularization
            l2_terms = [
                tf.nn.l2_loss(v)
                for v in train_vars
                if "bias" not in v.name.lower()
            ]

            if len(
                l2_terms
            ) > 0:

                loss_a = (
                    loss_a
                    +
                    UPA_WEIGHT_DECAY
                    * tf.add_n(
                        l2_terms
                    )
                )

        grads_a = tape_a.gradient(
            loss_a,
            train_vars
        )

        del weak_logits
        del strong_logits
        del tape_a

        # ====================================================
        # PART B:
        # SELECTED PSEUDO-LABEL CE
        # ====================================================
        rand_pos = np.random.randint(
            0,
            len(selected_indices_np),
            size=bsz
        )

        selected_batch_indices = (
            selected_indices_np[
                rand_pos
            ]
        )

        selected_paths = (
            upa_target_df.iloc[
                selected_batch_indices
            ]["file_path"]
            .astype(str)
            .values
        )

        selected_labels = tf.constant(
            epoch_pseudo[
                selected_batch_indices
            ],
            dtype=tf.int32
        )

        selected_imgs_list = []

        for path in selected_paths:

            img = decode_raw_image(
                path
            )

            img = weak_aug(
                img
            )

            selected_imgs_list.append(
                img
            )

        selected_imgs = tf.stack(
            selected_imgs_list,
            axis=0
        )

        with tf.GradientTape() as tape_b:

            _, selected_logits = upa_forward(
                selected_imgs,
                training=True
            )

            loss_cls = tf.reduce_mean(
                tf.keras.losses
                .sparse_categorical_crossentropy(
                    selected_labels,
                    selected_logits,
                    from_logits=True
                )
            )

            loss_b = (
                UPA_PAR_CLS
                * loss_cls
            )

        grads_b = tape_b.gradient(
            loss_b,
            train_vars
        )

        # ----------------------------------------------------
        # ADD BOTH GRADIENTS, THEN ONE OPTIMIZER STEP
        # ----------------------------------------------------
        combined_grads = (
            add_gradient_sets(
                grads_a,
                grads_b
            )
        )

        valid_pairs = [
            (g, v)
            for g, v in zip(
                combined_grads,
                train_vars
            )
            if g is not None
        ]

        optimizer_upa.apply_gradients(
            valid_pairs
        )

        total_loss_value = (
            float(
                loss_a.numpy()
            )
            +
            float(
                loss_b.numpy()
            )
        )

        epoch_total.append(
            total_loss_value
        )

        epoch_im.append(
            float(
                loss_im.numpy()
            )
        )

        epoch_cls.append(
            float(
                loss_cls.numpy()
            )
        )

        epoch_sucl.append(
            float(
                loss_sucl.numpy()
            )
        )

        del selected_logits
        del selected_imgs
        del selected_imgs_list
        del grads_a
        del grads_b
        del combined_grads
        del tape_b

    # ========================================================
    # EPOCH SUMMARY
    # ========================================================
    row = {
        "epoch":
            epoch,

        "total_loss":
            float(
                np.mean(
                    epoch_total
                )
            ),

        "im_loss":
            float(
                np.mean(
                    epoch_im
                )
            ),

        "cls_loss":
            float(
                np.mean(
                    epoch_cls
                )
            ),

        "sucl_loss":
            float(
                np.mean(
                    epoch_sucl
                )
            ),

        "selected":
            int(
                len(
                    selected_indices_np
                )
            ),

        "selected_fraction":
            float(
                len(
                    selected_indices_np
                )
                / len(
                    epoch_pseudo
                )
            ),

        "lr":
            float(
                optimizer_upa
                .learning_rate
                .numpy()
            )
    }

    upa_history.append(
        row
    )

    print(
        "\nEpoch summary:"
    )

    print(
        row
    )

    pd.DataFrame(
        upa_history
    ).to_csv(
        CONTROL_DIR
        / "upa_shared_training_history_seed42.csv",
        index=False
    )

    gc.collect()


# ============================================================
# 16. SAVE FINAL MODEL
# ============================================================

UPA_MODEL_PATH = (
    CONTROL_DIR
    / "ResNet50_UPA_shared_controlled_seed42.keras"
)

upa_model.save(
    UPA_MODEL_PATH
)

print(
    "\n"
    + "=" * 78
)

print(
    "CONTROLLED UPA TRAINING COMPLETE"
)

print(
    "=" * 78
)

print(
    "Saved model:",
    UPA_MODEL_PATH
)

print(
    "Saved history:",
    CONTROL_DIR
    / "upa_shared_training_history_seed42.csv"
)

Source checkpoint: /kaggle/working/external_sfuda_baseline/controlled_seed42/ResNet50_reproduced_source_seed42.keras
Target samples: 7200

CONTROLLED UPA MODEL LOADED
Backbone   : resnet50
Classifier : logits
Classifier frozen: True

Running memory-safe inference test...
Input: (8, 224, 224, 3)
Features: (8, 2048)
Logits: (8, 3)
UPA inference memory test PASSED.

UPA EPOCH 1/15

Pseudo-label counts:
  class 0: 2281
  class 1: 2451
  class 2: 2468
Starting UPA KNN pass 1/2
Starting UPA KNN pass 2/2

Selected: 4318 / 7200 (59.97%)

Epoch summary:
{'epoch': 1, 'total_loss': 2.866191363661214, 'im_loss': -1.0341900608274672, 'cls_loss': 0.15865733583331754, 'sucl_loss': 0.9556030711862776, 'selected': 4318, 'selected_fraction': 0.5997222222222223, 'lr': 0.0009890737710520625}

UPA EPOCH 2/15

Pseudo-label counts:
  class 0: 2267
  class 1: 2525
  class 2: 2408
Starting UPA KNN pass 1/2
Starting UPA KNN pass 2/2

Selected: 4319 / 7200 (59.99%)

Epoch summary:
{'epoch': 2, 'total_loss': 2.58

In [10]:
from IPython.display import Audio, display
import numpy as np

sr = 44100

def beep(freq, duration):
    t = np.linspace(0, duration, int(sr * duration), False)
    return 0.6 * np.sin(2 * np.pi * freq * t)

alarm = np.concatenate([
    beep(880, 0.5),
    np.zeros(int(sr * 0.2)),
    beep(1100, 0.5),
    np.zeros(int(sr * 0.2)),
    beep(880, 0.5),
    np.zeros(int(sr * 0.2)),
    beep(1320, 0.8),
])

display(Audio(alarm, rate=sr, autoplay=True))
print("🚨 UPA TRAINING FINISHED 🚨")

🚨 UPA TRAINING FINISHED 🚨


In [ ]:
# ============================================================
# POST-UPA: COMPLETE CONTROLLED SEED-42 EVALUATION + SAVE PACKAGE
# ============================================================

from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    log_loss,
)

from IPython.display import display, Audio


# ============================================================
# 0. LOCKED PATHS / SETTINGS
# ============================================================

CONTROL_DIR = Path(
    "/kaggle/working/external_sfuda_baseline/controlled_seed42"
)
CONTROL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_PACKAGE = Path(
    "/kaggle/input/datasets/dmmehedihasanabid/twd-kaggle/"
    "TWD_paper_final_package_FIXED_20260905_234741"
)

RAW_DIR = FINAL_PACKAGE / "raw_result_files"

SOURCE_MODEL_PATH = CONTROL_DIR / "ResNet50_reproduced_source_seed42.keras"

THREEWAY_MODEL_PATH = (
    CONTROL_DIR /
    "ResNet50_threeway_classaware_controlled_seed42.keras"
)

CONF_ACCEPT = 0.9999
CONF_REJECT = 0.6667

CLASS_NAMES = [
    "Early Blight",
    "Late Blight",
    "Healthy"
]

IMG_SIZE = 224
INFER_BATCH = 8

print("=" * 78)
print("POST-UPA CONTROLLED SEED-42 EVALUATION")
print("=" * 78)


# ============================================================
# 1. VERIFY TRAINING HISTORY
# ============================================================

history_obj = None

for candidate_name in [
    "upa_history",
    "history",
    "training_history",
]:
    if candidate_name in globals():
        candidate = globals()[candidate_name]

        if isinstance(candidate, list):
            history_obj = candidate
            break

        if hasattr(candidate, "history"):
            # Keras-style history
            temp = candidate.history
            if isinstance(temp, dict):
                history_obj = pd.DataFrame(temp)
            break


if isinstance(history_obj, list):
    history_df = pd.DataFrame(history_obj)

elif isinstance(history_obj, pd.DataFrame):
    history_df = history_obj.copy()

else:
    history_df = pd.DataFrame()


if len(history_df) > 0:

    print("\nTraining history rows:", len(history_df))

    if "epoch" in history_df.columns:
        print("Epoch values:", history_df["epoch"].tolist())

        if len(history_df) == 15:
            print("PASS: 15 training-history rows found.")
        else:
            print(
                "WARNING: expected 15 training-history rows, "
                f"found {len(history_df)}."
            )

    HISTORY_PATH = CONTROL_DIR / "upa_training_history_seed42.csv"
    history_df.to_csv(HISTORY_PATH, index=False)

    print("Saved:", HISTORY_PATH)

else:
    print(
        "\nWARNING: Could not automatically recover the in-memory "
        "history object. Evaluation will continue."
    )


# ============================================================
# 2. FIND / SAVE FINAL UPA MODEL
# ============================================================

UPA_MODEL_PATH = (
    CONTROL_DIR /
    "ResNet50_UPA_shared_backbone_controlled_seed42.keras"
)

upa_eval_model = None

# Prefer the actual model still in memory after training.
possible_model_names = [
    "upa_model",
    "model",
    "target_model",
    "student_model",
]

for model_name in possible_model_names:

    if model_name in globals():

        obj = globals()[model_name]

        if isinstance(obj, tf.keras.Model):

            # Avoid accidentally saving the source model.
            if obj is not globals().get("source_model", None):
                upa_eval_model = obj
                print(
                    "\nUsing in-memory UPA model:",
                    model_name
                )
                break


if upa_eval_model is not None:

    upa_eval_model.save(UPA_MODEL_PATH)
    print("Final UPA model saved:", UPA_MODEL_PATH)

else:

    # Search existing controlled-directory UPA models.
    candidates = sorted(
        list(CONTROL_DIR.glob("*UPA*.keras")) +
        list(CONTROL_DIR.glob("*upa*.keras"))
    )

    candidates = [
        p for p in candidates
        if "source" not in p.name.lower()
        and "threeway" not in p.name.lower()
    ]

    if len(candidates) == 0:
        raise RuntimeError(
            "Could not find the final UPA model in memory or on disk. "
            "Do NOT retrain. Inspect the final lines of the training cell "
            "and set UPA_MODEL_PATH to the model it saved."
        )

    UPA_MODEL_PATH = candidates[-1]

    print("\nLoading saved UPA model:", UPA_MODEL_PATH)

    upa_eval_model = tf.keras.models.load_model(
        UPA_MODEL_PATH,
        compile=False
    )


print("\nFinal UPA model path:")
print(UPA_MODEL_PATH)


# ============================================================
# 3. LOCATE EXACT LOCKED SEED-42 HARD TEST MANIFEST
# ============================================================

hard_test_candidates = list(
    RAW_DIR.rglob("target_hard_test_labeled.csv")
)

# Root raw_result_files file = original Seed-42 result.
seed42_hard_candidates = [
    p for p in hard_test_candidates
    if "seed_123" not in str(p)
    and "seed_2024" not in str(p)
]

if len(seed42_hard_candidates) == 0:
    raise FileNotFoundError(
        "Could not locate the exact saved Seed-42 "
        "target_hard_test_labeled.csv."
    )

# Prefer directly under raw_result_files.
root_candidates = [
    p for p in seed42_hard_candidates
    if p.parent == RAW_DIR
]

if len(root_candidates) > 0:
    HARD_TEST_CSV = root_candidates[0]
else:
    HARD_TEST_CSV = seed42_hard_candidates[0]


hard_test_df = pd.read_csv(HARD_TEST_CSV)

print("\nHard-target test manifest:")
print(HARD_TEST_CSV)
print("Rows:", len(hard_test_df))
print("Columns:", list(hard_test_df.columns))

assert len(hard_test_df) == 1800, (
    f"Expected 1800 hard-target Seed-42 test samples, "
    f"got {len(hard_test_df)}"
)


# ============================================================
# 4. INPUT COLUMN DETECTION
# ============================================================

PATH_CANDIDATES = [
    "file_path",
    "filepath",
    "path",
    "image_path",
]

LABEL_CANDIDATES = [
    "label",
    "label_id",
    "class_id",
    "target",
]

path_col = next(
    (c for c in PATH_CANDIDATES if c in hard_test_df.columns),
    None
)

label_col = next(
    (c for c in LABEL_CANDIDATES if c in hard_test_df.columns),
    None
)

if path_col is None:
    raise KeyError(
        f"Could not identify image-path column from "
        f"{list(hard_test_df.columns)}"
    )

if label_col is None:
    raise KeyError(
        f"Could not identify label column from "
        f"{list(hard_test_df.columns)}"
    )

print("\nUsing path column :", path_col)
print("Using label column:", label_col)


# ============================================================
# 5. EXACT INFERENCE DATA PIPELINE
# ============================================================

def load_eval_image(path, label):

    img = tf.io.read_file(path)

    img = tf.io.decode_image(
        img,
        channels=3,
        expand_animations=False
    )

    img.set_shape([None, None, 3])

    img = tf.image.resize(
        img,
        [IMG_SIZE, IMG_SIZE]
    )

    img = tf.cast(img, tf.float32)

    return img, label


def make_eval_ds(df):

    paths = df[path_col].astype(str).values
    labels = df[label_col].astype(np.int32).values

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    ds = ds.map(
        load_eval_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(INFER_BATCH)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


hard_test_ds = make_eval_ds(hard_test_df)


# ============================================================
# 6. METRIC FUNCTIONS
# ============================================================

def stable_softmax(logits):

    logits = np.asarray(logits, dtype=np.float64)

    logits = logits - logits.max(
        axis=1,
        keepdims=True
    )

    exps = np.exp(logits)

    return exps / exps.sum(
        axis=1,
        keepdims=True
    )


def expected_calibration_error(
    y_true,
    probs,
    n_bins=15
):

    probs = np.asarray(probs)

    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)

    correctness = (
        predictions == np.asarray(y_true)
    ).astype(float)

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    ece = 0.0

    for i in range(n_bins):

        lo = bin_edges[i]
        hi = bin_edges[i + 1]

        if i == n_bins - 1:
            mask = (
                (confidences >= lo)
                & (confidences <= hi)
            )
        else:
            mask = (
                (confidences >= lo)
                & (confidences < hi)
            )

        if mask.any():

            bin_acc = correctness[mask].mean()
            bin_conf = confidences[mask].mean()

            ece += (
                mask.mean()
                * abs(bin_acc - bin_conf)
            )

    return float(ece)


def multiclass_brier(y_true, probs, n_classes=3):

    y_true = np.asarray(
        y_true,
        dtype=np.int64
    )

    onehot = np.eye(
        n_classes,
        dtype=np.float64
    )[y_true]

    return float(
        np.mean(
            np.sum(
                (probs - onehot) ** 2,
                axis=1
            )
        )
    )


def evaluate_model(model, ds, name):

    y_true = []
    logits_all = []

    for xb, yb in ds:

        logits = model(
            xb,
            training=False
        ).numpy()

        logits_all.append(logits)
        y_true.append(yb.numpy())

    y_true = np.concatenate(y_true)
    logits_all = np.concatenate(logits_all)

    probs = stable_softmax(logits_all)
    preds = probs.argmax(axis=1)

    metrics = {
        "Method": name,
        "Accuracy": accuracy_score(
            y_true,
            preds
        ),
        "Macro_F1": f1_score(
            y_true,
            preds,
            average="macro"
        ),
        "ECE": expected_calibration_error(
            y_true,
            probs,
            n_bins=15
        ),
        "Brier": multiclass_brier(
            y_true,
            probs,
            n_classes=3
        ),
        "NLL": log_loss(
            y_true,
            probs,
            labels=[0, 1, 2]
        ),
    }

    cm = confusion_matrix(
        y_true,
        preds,
        labels=[0, 1, 2]
    )

    return (
        metrics,
        y_true,
        logits_all,
        probs,
        preds,
        cm
    )


# ============================================================
# 7. EVALUATE UPA
# ============================================================

print("\n" + "=" * 78)
print("EVALUATING UPA ON LOCKED SEED-42 HARD TARGET TEST")
print("=" * 78)

(
    upa_metrics,
    y_true,
    upa_logits,
    upa_probs,
    upa_preds,
    upa_cm
) = evaluate_model(
    upa_eval_model,
    hard_test_ds,
    "UPA (shared-backbone)"
)

print("\nUPA HARD-TARGET RESULTS")
for k, v in upa_metrics.items():
    if k != "Method":
        print(f"{k:10s}: {v:.6f}")


print("\nUPA CONFUSION MATRIX")
print(upa_cm)


# ============================================================
# 8. SAVE SAMPLE-LEVEL UPA PREDICTIONS
# ============================================================

upa_prediction_df = hard_test_df.copy()

upa_prediction_df["true_label_eval"] = y_true
upa_prediction_df["pred_label"] = upa_preds
upa_prediction_df["confidence"] = upa_probs.max(axis=1)

for c in range(3):
    upa_prediction_df[
        f"prob_class_{c}"
    ] = upa_probs[:, c]


UPA_PRED_PATH = (
    CONTROL_DIR /
    "upa_seed42_hard_target_predictions.csv"
)

upa_prediction_df.to_csv(
    UPA_PRED_PATH,
    index=False
)

print("\nSaved predictions:", UPA_PRED_PATH)


# ============================================================
# 9. CONTROLLED COMPARISON TABLE
# ============================================================

controlled_rows = [
    {
        "Method": "SourceOnly",
        "Accuracy": 0.840000,
        "Macro_F1": 0.834886,
        "ECE": 0.106917,
        "Brier": 0.261456,
        "NLL": 0.730451,
    },
    {
        "Method": "ThreeWay-ClassAware",
        "Accuracy": 0.843333,
        "Macro_F1": 0.838457,
        "ECE": 0.046619,
        "Brier": 0.233041,
        "NLL": 0.459743,
    },
    upa_metrics,
]

controlled_df = pd.DataFrame(
    controlled_rows
)

CONTROLLED_RESULTS_PATH = (
    CONTROL_DIR /
    "controlled_seed42_source_threeway_upa.csv"
)

controlled_df.to_csv(
    CONTROLLED_RESULTS_PATH,
    index=False
)

print("\n" + "=" * 78)
print("CONTROLLED SEED-42 COMPARISON")
print("=" * 78)

display(
    controlled_df.style.format({
        "Accuracy": "{:.4f}",
        "Macro_F1": "{:.4f}",
        "ECE": "{:.4f}",
        "Brier": "{:.4f}",
        "NLL": "{:.4f}",
    })
)


# ============================================================
# 10. APPLY SAME FIXED THREE-WAY EVALUATION GATE TO UPA
# ============================================================

upa_conf = upa_probs.max(axis=1)

upa_region = np.where(
    upa_conf >= CONF_ACCEPT,
    "Accept",
    np.where(
        upa_conf < CONF_REJECT,
        "Reject",
        "Defer"
    )
)

region_rows = []

for region_name in [
    "Accept",
    "Defer",
    "Reject"
]:

    mask = upa_region == region_name

    count = int(mask.sum())

    coverage = count / len(upa_region)

    if count > 0:

        acc = float(
            np.mean(
                upa_preds[mask]
                == y_true[mask]
            )
        )

        mean_conf = float(
            upa_conf[mask].mean()
        )

    else:

        acc = np.nan
        mean_conf = np.nan

    region_rows.append({
        "Region": region_name,
        "Count": count,
        "Coverage": coverage,
        "Accuracy": acc,
        "MeanConfidence": mean_conf,
    })


upa_region_df = pd.DataFrame(
    region_rows
)

UPA_REGION_PATH = (
    CONTROL_DIR /
    "upa_seed42_threeway_evaluation_regions.csv"
)

upa_region_df.to_csv(
    UPA_REGION_PATH,
    index=False
)

print("\n" + "=" * 78)
print("UPA UNDER THE SAME FIXED THREE-WAY EVALUATION GATE")
print("=" * 78)

display(
    upa_region_df.style.format({
        "Coverage": "{:.4f}",
        "Accuracy": "{:.4f}",
        "MeanConfidence": "{:.4f}",
    })
)


# Add region to predictions.
upa_prediction_df["eval_region"] = upa_region

upa_prediction_df.to_csv(
    UPA_PRED_PATH,
    index=False
)


# ============================================================
# 11. CONFUSION-MATRIX FIGURE
# ============================================================

fig = plt.figure(
    figsize=(5.2, 4.5)
)

ax = fig.add_subplot(111)

im = ax.imshow(upa_cm)

ax.set_xticks(range(3))
ax.set_yticks(range(3))

ax.set_xticklabels(
    CLASS_NAMES,
    rotation=35,
    ha="right"
)

ax.set_yticklabels(CLASS_NAMES)

ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title(
    "UPA (shared-backbone): Seed-42 hard target"
)

for i in range(3):
    for j in range(3):

        ax.text(
            j,
            i,
            str(upa_cm[i, j]),
            ha="center",
            va="center"
        )

fig.tight_layout()

UPA_CM_PNG = (
    CONTROL_DIR /
    "upa_seed42_confusion_matrix.png"
)

UPA_CM_PDF = (
    CONTROL_DIR /
    "upa_seed42_confusion_matrix.pdf"
)

fig.savefig(
    UPA_CM_PNG,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    UPA_CM_PDF,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 12. CONTROLLED PERFORMANCE FIGURE
# ============================================================

metric_plot_df = controlled_df.set_index(
    "Method"
)[[
    "Accuracy",
    "Macro_F1"
]]

ax = metric_plot_df.plot(
    kind="bar",
    figsize=(7.0, 4.3)
)

ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.set_title(
    "Controlled Seed-42 hard-target performance"
)
ax.legend(loc="lower right")

plt.xticks(
    rotation=15,
    ha="right"
)

plt.tight_layout()

PERF_PNG = (
    CONTROL_DIR /
    "controlled_seed42_accuracy_macrof1.png"
)

PERF_PDF = (
    CONTROL_DIR /
    "controlled_seed42_accuracy_macrof1.pdf"
)

plt.savefig(
    PERF_PNG,
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    PERF_PDF,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 13. CONTROLLED RELIABILITY FIGURE
# ============================================================

reliability_plot_df = controlled_df.set_index(
    "Method"
)[[
    "ECE",
    "Brier",
    "NLL"
]]

ax = reliability_plot_df.plot(
    kind="bar",
    figsize=(7.0, 4.3)
)

ax.set_ylabel("Error / Loss")
ax.set_title(
    "Controlled Seed-42 reliability comparison"
)

plt.xticks(
    rotation=15,
    ha="right"
)

plt.tight_layout()

REL_PNG = (
    CONTROL_DIR /
    "controlled_seed42_reliability.png"
)

REL_PDF = (
    CONTROL_DIR /
    "controlled_seed42_reliability.pdf"
)

plt.savefig(
    REL_PNG,
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    REL_PDF,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 14. TRAINING-HISTORY FIGURE
# ============================================================

if len(history_df) > 0:

    possible_loss_cols = [
        c for c in [
            "total_loss",
            "im_loss",
            "cls_loss",
            "sucl_loss"
        ]
        if c in history_df.columns
    ]

    if len(possible_loss_cols) > 0:

        x = (
            history_df["epoch"].values
            if "epoch" in history_df.columns
            else np.arange(
                1,
                len(history_df) + 1
            )
        )

        fig = plt.figure(
            figsize=(7.0, 4.3)
        )

        ax = fig.add_subplot(111)

        for col in possible_loss_cols:
            ax.plot(
                x,
                history_df[col],
                marker="o",
                label=col
            )

        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title(
            "UPA controlled Seed-42 training history"
        )

        ax.legend()
        fig.tight_layout()

        HIST_PNG = (
            CONTROL_DIR /
            "upa_seed42_training_history.png"
        )

        HIST_PDF = (
            CONTROL_DIR /
            "upa_seed42_training_history.pdf"
        )

        fig.savefig(
            HIST_PNG,
            dpi=300,
            bbox_inches="tight"
        )

        fig.savefig(
            HIST_PDF,
            bbox_inches="tight"
        )

        plt.show()


# ============================================================
# 15. OPTIONAL OOD FALSE-ACCEPT EVALUATION
# ============================================================

print("\n" + "=" * 78)
print("SEARCHING FOR SAVED OOD MANIFEST")
print("=" * 78)

ood_candidates = []

for pattern in [
    "*ood*.csv",
    "*OOD*.csv",
]:

    ood_candidates.extend(
        RAW_DIR.rglob(pattern)
    )


# Remove obvious summary/result CSVs where possible.
ood_manifest_candidates = [
    p for p in ood_candidates
    if "summary" not in p.name.lower()
    and "comparison" not in p.name.lower()
    and "result" not in p.name.lower()
]


print("Possible OOD files:")
for p in ood_manifest_candidates[:30]:
    print(" -", p)


def looks_like_image_manifest(path):

    try:
        temp = pd.read_csv(
            path,
            nrows=5
        )

        return any(
            c in temp.columns
            for c in PATH_CANDIDATES
        )

    except Exception:
        return False


ood_image_manifests = [
    p for p in ood_manifest_candidates
    if looks_like_image_manifest(p)
]

# Prefer root / Seed-42 files.
ood_seed42_candidates = [
    p for p in ood_image_manifests
    if "seed_123" not in str(p)
    and "seed_2024" not in str(p)
]


if len(ood_seed42_candidates) > 0:

    OOD_CSV = ood_seed42_candidates[0]

    ood_df = pd.read_csv(OOD_CSV)

    ood_path_col = next(
        (
            c for c in PATH_CANDIDATES
            if c in ood_df.columns
        ),
        None
    )

    if (
        ood_path_col is not None
        and len(ood_df) == 2000
    ):

        print(
            "\nUsing OOD manifest:",
            OOD_CSV
        )

        print(
            "OOD rows:",
            len(ood_df)
        )

        def load_ood_image(path):

            img = tf.io.read_file(path)

            img = tf.io.decode_image(
                img,
                channels=3,
                expand_animations=False
            )

            img.set_shape(
                [None, None, 3]
            )

            img = tf.image.resize(
                img,
                [IMG_SIZE, IMG_SIZE]
            )

            return tf.cast(
                img,
                tf.float32
            )


        ood_ds = tf.data.Dataset.from_tensor_slices(
            ood_df[
                ood_path_col
            ].astype(str).values
        )

        ood_ds = ood_ds.map(
            load_ood_image,
            num_parallel_calls=tf.data.AUTOTUNE
        )

        ood_ds = ood_ds.batch(
            INFER_BATCH
        ).prefetch(
            tf.data.AUTOTUNE
        )


        ood_logits_list = []

        for xb in ood_ds:

            logits = upa_eval_model(
                xb,
                training=False
            ).numpy()

            ood_logits_list.append(
                logits
            )


        ood_logits = np.concatenate(
            ood_logits_list
        )

        ood_probs = stable_softmax(
            ood_logits
        )

        ood_conf = ood_probs.max(axis=1)

        ood_accept = (
            ood_conf >= CONF_ACCEPT
        )

        ood_defer = (
            (ood_conf >= CONF_REJECT)
            & (ood_conf < CONF_ACCEPT)
        )

        ood_reject = (
            ood_conf < CONF_REJECT
        )

        ood_result = pd.DataFrame([{
            "Method":
                "UPA (shared-backbone)",
            "OOD_N":
                len(ood_df),
            "Accept_Count":
                int(ood_accept.sum()),
            "False_Accept_Rate":
                float(ood_accept.mean()),
            "Defer_Count":
                int(ood_defer.sum()),
            "Defer_Rate":
                float(ood_defer.mean()),
            "Reject_Count":
                int(ood_reject.sum()),
            "Reject_Rate":
                float(ood_reject.mean()),
        }])


        OOD_RESULT_PATH = (
            CONTROL_DIR /
            "upa_seed42_ood_fixed_gate.csv"
        )

        ood_result.to_csv(
            OOD_RESULT_PATH,
            index=False
        )

        print(
            "\nUPA OOD FIXED-GATE RESULTS"
        )

        display(
            ood_result.style.format({
                "False_Accept_Rate":
                    "{:.4%}",
                "Defer_Rate":
                    "{:.4%}",
                "Reject_Rate":
                    "{:.4%}",
            })
        )

    else:

        print(
            "\nOOD candidate found, but it did "
            "not match the expected 2,000-image "
            "manifest. OOD evaluation skipped."
        )

else:

    print(
        "\nNo unambiguous 2,000-image OOD "
        "manifest was found automatically."
    )

    print(
        "This does NOT affect the hard-target "
        "UPA evaluation."
    )


# ============================================================
# 16. SAVE MACHINE-READABLE SUMMARY
# ============================================================

summary = {
    "experiment":
        "Controlled Seed-42 UPA shared-backbone comparison",

    "upa_model_path":
        str(UPA_MODEL_PATH),

    "hard_test_manifest":
        str(HARD_TEST_CSV),

    "hard_test_n":
        int(len(hard_test_df)),

    "accept_threshold":
        CONF_ACCEPT,

    "reject_threshold":
        CONF_REJECT,

    "upa_metrics": {
        k: (
            float(v)
            if isinstance(
                v,
                (
                    float,
                    np.floating,
                    int,
                    np.integer,
                )
            )
            else v
        )
        for k, v in upa_metrics.items()
    },

    "notes": [
        "Controlled Seed-42 comparison only.",
        "UPA is a shared-backbone implementation.",
        "Same reproduced source initialization is used for controlled comparison.",
        "The fixed three-way gate is an evaluation gate and is not part of UPA training.",
        "No target-test labels were used for model selection during UPA adaptation."
    ],
}


SUMMARY_JSON = (
    CONTROL_DIR /
    "controlled_seed42_upa_summary.json"
)

with open(
    SUMMARY_JSON,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

print("\nSaved:", SUMMARY_JSON)


# ============================================================
# 17. ZIP EVERYTHING FOR TOMORROW
# ============================================================

ZIP_PATH = Path(
    "/kaggle/working/"
    "controlled_seed42_UPA_COMPLETE_RESULTS.zip"
)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for path in CONTROL_DIR.rglob("*"):

        if path.is_file():

            zf.write(
                path,
                arcname=path.relative_to(
                    CONTROL_DIR.parent
                )
            )


print("\n" + "=" * 78)
print("FINAL OUTPUT PACKAGE CREATED")
print("=" * 78)

print(ZIP_PATH)


# ============================================================
# 18. FINAL SCREEN SUMMARY
# ============================================================

print("\n\n")
print("=" * 78)
print("FINAL CONTROLLED SEED-42 RESULT")
print("=" * 78)

print(
    controlled_df.to_string(
        index=False
    )
)

print("\nUPA three-way evaluation regions:")
print(
    upa_region_df.to_string(
        index=False
    )
)

print("\nArtifacts saved under:")
print(CONTROL_DIR)

print("\nZIP:")
print(ZIP_PATH)

print("\n🚨 POST-UPA EVALUATION FINISHED 🚨")


# ============================================================
# 19. ALARM
# ============================================================

sr = 44100

def beep(freq, duration):

    t = np.linspace(
        0,
        duration,
        int(sr * duration),
        False
    )

    return (
        0.6
        * np.sin(
            2 * np.pi * freq * t
        )
    )


alarm = np.concatenate([
    beep(880, 0.45),
    np.zeros(int(sr * 0.15)),
    beep(1100, 0.45),
    np.zeros(int(sr * 0.15)),
    beep(880, 0.45),
    np.zeros(int(sr * 0.15)),
    beep(1320, 0.80),
])

display(
    Audio(
        alarm,
        rate=sr,
        autoplay=True
    )
)

In [ ]:
# ============================================================
# KEEP KAGGLE SESSION ACTIVE OVERNIGHT
# Run this AFTER all evaluation / saving cells finish
# ============================================================

import time
from datetime import datetime

HOURS = 8
INTERVAL_MINUTES = 5

total_steps = int(HOURS * 60 / INTERVAL_MINUTES)

print("=" * 70)
print("OVERNIGHT KEEP-ALIVE STARTED")
print("Start time:", datetime.now())
print(f"Planned duration: {HOURS} hours")
print("=" * 70)

for i in range(total_steps):
    print(
        f"[{datetime.now()}] "
        f"Keep-alive {i+1}/{total_steps} — "
        "results already saved in /kaggle/working"
    )
    time.sleep(INTERVAL_MINUTES * 60)

print("=" * 70)
print("KEEP-ALIVE FINISHED")
print("Finish time:", datetime.now())
print("=" * 70)

In [ ]:
# ============================================================
# FINAL NOTEBOOK DISPLAY — ALL CONTROLLED SEED-42 RESULTS
# Run AFTER the evaluation block
# ============================================================

from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

CONTROL_DIR = Path(
    "/kaggle/working/external_sfuda_baseline/controlled_seed42"
)

ZIP_PATH = Path(
    "/kaggle/working/controlled_seed42_UPA_COMPLETE_RESULTS.zip"
)

print("=" * 90)
print("FINAL CONTROLLED SEED-42 RESULTS DASHBOARD")
print("=" * 90)


# ------------------------------------------------------------
# 1. UPA TRAINING HISTORY
# ------------------------------------------------------------

history_candidates = [
    CONTROL_DIR / "upa_shared_training_history_seed42.csv",
    CONTROL_DIR / "upa_training_history_seed42.csv",
]

history_path = next(
    (p for p in history_candidates if p.exists()),
    None
)

if history_path is not None:
    history_df = pd.read_csv(history_path)

    display(Markdown("## 1. UPA Training History"))

    print("History file:", history_path)
    print("Rows:", len(history_df))

    display(history_df)

else:
    print("\nWARNING: UPA history CSV not found.")


# ------------------------------------------------------------
# 2. CONTROLLED SOURCE vs THREE-WAY vs UPA TABLE
# ------------------------------------------------------------

comparison_path = (
    CONTROL_DIR /
    "controlled_seed42_source_threeway_upa.csv"
)

display(Markdown("## 2. Controlled Seed-42 Comparison"))

if comparison_path.exists():

    comparison_df = pd.read_csv(comparison_path)

    display(
        comparison_df.style
        .format({
            "Accuracy": "{:.4f}",
            "Macro_F1": "{:.4f}",
            "ECE": "{:.4f}",
            "Brier": "{:.4f}",
            "NLL": "{:.4f}",
        })
    )

    print("\nRaw values:")
    print(comparison_df.to_string(index=False))

else:
    print("Comparison table not found.")


# ------------------------------------------------------------
# 3. UPA THREE-WAY REGION RESULTS
# ------------------------------------------------------------

region_path = (
    CONTROL_DIR /
    "upa_seed42_threeway_evaluation_regions.csv"
)

display(Markdown("## 3. UPA Three-Way Evaluation Regions"))

if region_path.exists():

    region_df = pd.read_csv(region_path)

    display(
        region_df.style
        .format({
            "Coverage": "{:.4f}",
            "Accuracy": "{:.4f}",
            "MeanConfidence": "{:.4f}",
        })
    )

    print("\nRaw values:")
    print(region_df.to_string(index=False))

else:
    print("Three-way region file not found.")


# ------------------------------------------------------------
# 4. UPA OOD RESULTS
# ------------------------------------------------------------

ood_path = (
    CONTROL_DIR /
    "upa_seed42_ood_fixed_gate.csv"
)

display(Markdown("## 4. UPA OOD Fixed-Gate Evaluation"))

if ood_path.exists():

    ood_df = pd.read_csv(ood_path)

    display(
        ood_df.style
        .format({
            "False_Accept_Rate": "{:.4%}",
            "Defer_Rate": "{:.4%}",
            "Reject_Rate": "{:.4%}",
        })
    )

    print("\nRaw values:")
    print(ood_df.to_string(index=False))

else:
    print(
        "OOD result was not generated, most likely because "
        "the exact OOD manifest was not automatically identified."
    )


# ------------------------------------------------------------
# 5. MACHINE-READABLE SUMMARY JSON
# ------------------------------------------------------------

summary_path = (
    CONTROL_DIR /
    "controlled_seed42_upa_summary.json"
)

display(Markdown("## 5. Saved Experiment Summary"))

if summary_path.exists():

    with open(summary_path, "r") as f:
        summary = json.load(f)

    print(json.dumps(summary, indent=2))

else:
    print("Summary JSON not found.")


# ------------------------------------------------------------
# 6. SAVED FIGURES
# ------------------------------------------------------------

display(Markdown("## 6. Saved Figures"))

figure_files = [
    (
        "UPA Confusion Matrix",
        CONTROL_DIR / "upa_seed42_confusion_matrix.png"
    ),
    (
        "Accuracy and Macro-F1",
        CONTROL_DIR / "controlled_seed42_accuracy_macrof1.png"
    ),
    (
        "Reliability Metrics",
        CONTROL_DIR / "controlled_seed42_reliability.png"
    ),
    (
        "UPA Training History",
        CONTROL_DIR / "upa_seed42_training_history.png"
    ),
]

for title, fig_path in figure_files:

    if fig_path.exists():

        display(Markdown(f"### {title}"))
        display(Image(filename=str(fig_path)))

    else:
        print(f"Not found: {fig_path.name}")


# ------------------------------------------------------------
# 7. LIST ALL SAVED ARTIFACTS
# ------------------------------------------------------------

display(Markdown("## 7. Saved Files"))

saved_files = sorted(
    [
        p for p in CONTROL_DIR.rglob("*")
        if p.is_file()
    ]
)

for p in saved_files:

    size_mb = p.stat().st_size / (1024 ** 2)

    print(
        f"{p.name:<65} "
        f"{size_mb:>9.2f} MB"
    )


# ------------------------------------------------------------
# 8. CHECK CRITICAL FILES
# ------------------------------------------------------------

display(Markdown("## 8. Critical File Check"))

critical_files = {
    "Final UPA model":
        CONTROL_DIR /
        "ResNet50_UPA_shared_controlled_seed42.keras",

    "UPA history":
        CONTROL_DIR /
        "upa_shared_training_history_seed42.csv",

    "Controlled comparison":
        CONTROL_DIR /
        "controlled_seed42_source_threeway_upa.csv",

    "UPA predictions":
        CONTROL_DIR /
        "upa_seed42_hard_target_predictions.csv",

    "Three-way regions":
        CONTROL_DIR /
        "upa_seed42_threeway_evaluation_regions.csv",

    "Summary JSON":
        CONTROL_DIR /
        "controlled_seed42_upa_summary.json",

    "Final ZIP":
        ZIP_PATH,
}


all_ok = True

for name, path in critical_files.items():

    exists = path.exists()

    status = "FOUND" if exists else "MISSING"

    print(
        f"{status:8s} | {name}"
    )

    if not exists:
        all_ok = False


# ------------------------------------------------------------
# 9. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 90)

if all_ok:
    print("SUCCESS: ALL CRITICAL RESULTS AND FILES ARE PRESENT.")
else:
    print(
        "WARNING: ONE OR MORE CRITICAL FILES ARE MISSING. "
        "CHECK THE OUTPUT ABOVE."
    )

print("=" * 90)

print("\nMain results folder:")
print(CONTROL_DIR)

print("\nFinal ZIP:")
print(ZIP_PATH)

print("\nYou can now start the overnight keep-alive cell.")